In [3]:
import sys 
sys.path.append('C:/Users/Data/Documents/lazo_fernando/target_script_01/funciones')
from variables_inicio import *
from sqlalchemy import create_engine
from sqlalchemy import text

import numpy as np
from funciones import *
from funciones_spark import *
from utils_sql import *

spark = SparkSession.builder \
    .appName("SparkExample") \
    .master("local[*]") \
    .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.4.0.jre11.jar') \
    .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.4.0.jre11.jar') \
    .config('spark.executor.memory', '8g') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

engine_mysql = create_engine(
    f"mysql+pymysql://{user_envio}:{pwd_envio}@{server_envio}:{port_mysql}/{db_envio}"
)


In [2]:
query = f"""
select * from Alice.prospectos_correos_alfin
where fecha_registro >='2026-08-01'
"""
df_correo = pd.read_sql(query, engine_mysql)
# query = f"""
# select dni_cliente, fecha_visita, DATE(fecha_envio) AS fecha_envio 
# from Alice.prospectos_envio_alfin
# where fecha_creacion >='2026-08-01'
# and estado='PROCESADO'
# """
# df_formulario = pd.read_sql(query, engine_mysql)


In [37]:
display(df_correo.head(2))
display(df_formulario.head(2))

,dni_cliente,nombre_cliente,celular,fecha_visita,hora_visita,fecha_envio
0,48124718,CHOQUE CURO NOEMY LUZMERY,937616610,2026-08-04,0 days 11:30:00,2026-08-01
1,41494656,GUTIERREZ HEREDIA CARLOS ALBERTO,994746090,2026-08-06,0 days 12:00:00,2026-08-01


,dni_cliente,fecha_visita,fecha_envio
0,80644018,2026-08-05,2026-08-02
1,80632938,2026-08-02,2026-08-01


In [ ]:
df_correo=df_correo.merge(df_formulario,on=['dni_cliente','fecha_envio'],how='inner')

In [42]:
df_group = (
    df_correo
    .groupby(["fecha_envio"])
    .size()
    .reset_index(name="cantidad")
)
df_group.head()

,fecha_envio,cantidad
0,2026-08-01,2206
1,2026-08-02,2714


In [19]:
print(df_formulario.columns.tolist())

['id', 'hash_duplicado', 'dni_vendedor', 'operador', 'dni_cliente', 'nombre_cliente', 'telefono_cliente', 'agencia_tienda', 'fecha_visita', 'monto_solicitado', 'tipo_gestion', 'estado', 'codigo_http_ms', 'respuesta_ms', 'fecha_creacion', 'fecha_envio']


In [43]:

ruta_archivo = os.path.join(ruta_alfin, 'usar_01.csv')
df_correo.to_csv(ruta_archivo, sep=';')

In [2]:
from pyspark.sql import functions as F

def completar_con_ceros(df, columna, longitud=8):
    return df.withColumn(
        columna,
        F.lpad(F.col(columna).cast("string"), longitud, "0")
    )


In [ ]:
filename='REPRICING CANALES EXTERNOS_20260822.csv'
df_lista=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)

filename='Consulta_de_Campañas_202608_V3_SS (RED_CALL)db2.csv'
df_validar_01=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)
filename='Consulta_de_Campañas_202608_V3_SS (RED_CALL)db.csv'
df_validar_02=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)
filename='Consulta_de_Campañas_202608_V5_SS_EXT (CAMPO)_db2.csv'
df_validar_03=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)
filename='Consulta_de_Campañas_202608_V5_SS_EXT (CAMPO)_db.csv'
df_validar_04=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)
df_validar_01=df_validar_01.drop('TASA_MIN_DESCUENTO')
df_validar_02=df_validar_02.drop('TASA_MIN_DESCUENTO')
df_validar_01=df_validar_01.withColumn('tipo_archivo',F.lit('campo'))
df_validar_02=df_validar_02.withColumn('tipo_archivo',F.lit('campo'))
df_validar_03=df_validar_03.withColumn('tipo_archivo',F.lit('call'))
df_validar_04=df_validar_04.withColumn('tipo_archivo',F.lit('call'))

df_validar=df_validar_01.unionByName(df_validar_02).unionByName(df_validar_03).unionByName(df_validar_04)

print(df_lista.columns)
print(df_validar.columns)


['NumeroDocumento', 'Agencia', 'TASA_NUEVA']
['DNI', 'COLOR_FINAL', 'COD_USER_V3', 'USER_V3', 'PERFIL_RO', 'campaña', 'OFERTA_MAX', 'PLAZO', 'CAPACIDAD_MAX', 'FRESCURA', 'rango_deuda', 'numentidades', 'TOTAL_A_LIQUIDAR', 'TASA_CREDITO_ANTERIOR', 'TASA_1', 'TASA_2', 'TASA_3', 'TASA_4', 'TASA_5', 'TASA_6', 'TASA_7', 'MGNEG', 'MARCA_PD', 'AUTORIZACION_DATOS', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'TIPO_BASE', 'PROPENSION_DISTRIBUCION', 'OFERTA_SS', 'TASA_1_SS', 'TASA_2_SS', 'TASA_3_SS', 'TASA_4_SS', 'TASA_5_SS', 'TASA_6_SS', 'TASA_7_SS', 'ALERTA_MAQUETA', 'FEN', 'PERFIL_ESPECIAL', 'TIPO', 'tipo_archivo']


In [ ]:
02800922

In [21]:
df_lista=df_lista.withColumnRenamed('NumeroDocumento','DNI')

In [22]:
query = """
select  NUMERO_DOCUMENTO as DNI,Agencia_comercial,cl_telf1 from DANTALION.dbo.Base_Maestra_ALFIN_BK
where cl_telf1 is not null
and retiro='ACTIVO'
    """
df_formato=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)

In [23]:
df_lista_unica=df_lista.join(df_formato,['DNI'],'inner')
df_lista_unica=df_lista_unica.join(df_validar,['DNI'],'inner')


In [25]:
df_lista_unica=df_lista_unica.dropDuplicates(['DNI'])

In [24]:
df_lista_unica.dropDuplicates(['DNI']).count()

23616

In [26]:
df_lista_pd=df_lista_unica.toPandas()

In [27]:
filename='TARGET.txt'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_target_desembolso = pd.read_csv(ruta_archivo,sep='|')
df_target_desembolso = df_target_desembolso[['DNI']].copy()
df_target_desembolso['CANAL']='CANAL'
filename='ACUM_DESEM.txt'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_fugas = pd.read_csv(ruta_archivo,sep='|')
df_fugas = df_fugas[['DNI','CANALVENTA']].copy()
df_target_desembolso['DNI'] = (
    df_target_desembolso['DNI']
    .astype(str)
    .str.replace(r'\D', '', regex=True)   
    .replace('', pd.NA)                     
    .str.zfill(8)                           
)
df_fugas['DNI'] = (
    df_fugas['DNI']
    .astype(str)
    .str.replace(r'\D', '', regex=True)   
    .replace('', pd.NA)                     
    .str.zfill(8)                           
)

df_desembolso=df_fugas.merge(
    df_target_desembolso,
    on=['DNI'],
    how='left'
)
df_desembolso = df_desembolso.fillna("OTROS")
df_desembolso.rename(columns={'DNI': 'dni_cliente'}, inplace=True)


filename='RetiroDefinitivo_BlackList.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_def_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_BlackList.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_Telefonos.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_telf = pd.read_csv(ruta_archivo,sep='|')
filename='retiro_correo_alfin.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_retiro_correo = pd.read_csv(ruta_archivo,sep=';')

# filename='desembolso.csv'
# ruta_archivo = os.path.join(ruta_alfin, filename)
# df_des = pd.read_csv(ruta_archivo,sep=';')

df_def_blacklist = df_def_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_blacklist = df_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_telf= df_telf.rename(columns={'TELEFONO': 'celular'})
df_retiro_correo= df_retiro_correo.rename(columns={'DNI': 'dni_cliente'})

# Blacklists de DNI
# df6 = df_des.copy()
# df6["celular"] = None
# df6 = df6[["dni_cliente", "celular"]]

# Blacklists de DNI
df1 = df_def_blacklist.copy()
df1["celular"] = None
df1 = df1[["dni_cliente", "celular"]]

df2 = df_blacklist.copy()
df2["celular"] = None
df2 = df2[["dni_cliente", "celular"]]

# Blacklist de teléfonos
df3 = df_telf.copy()
df3["dni_cliente"] = None
df3 = df3[["dni_cliente", "celular"]]

# Archivo con DNI y celular
df4 = df_retiro_correo[["dni_cliente", "celular"]].copy()

# Unir todo
df_retiros = pd.concat(
    [df1, df2, df3, df4],
    ignore_index=True
)



dni_retiro = set(df_retiros['dni_cliente'].dropna())
cel_retiro = set(df_retiros['celular'].dropna())
dni_desembolso = set(df_desembolso['dni_cliente'].dropna())


server_sql = server_zeus
db_sql = "THOTH"
user_sql = user_zeus
pwd_sql = pwd_zeus

engine = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
query = f"""
	SELECT distinct Dni,Descripcion_ FROM THOTH.dbo.Tmp_LLamadas_Alfin 
    where Descripcion_ in(
    'TELEFONO FUERA DE SERVICIO / NO EXISTE',
       'EXPRESO RECIBIR MÚLTIPLES LLAMADAS',
       'SOLICITÓ NO SER CONTACTADO',
       'FUERA DE SERVICIO',
       'EXPRESO FUTURA DENUNCIA ANTE INDECOPI O REGULADOR',
       'EXPRESO QUE NO AUTORIZÓ USO DE DATOS PERSONALES'
    )
"""
df_tipis = pd.read_sql(query, engine)
set_tipi = set(
    df_tipis['Dni']
    .dropna()
    .drop_duplicates()
)



C:\Users\DATA\AppData\Local\Temp\ipykernel_19364\3884659183.py:79: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_retiros = pd.concat(


In [29]:
df_lista_pd=df_lista_pd[
    ~df_lista_pd['DNI'].isin(dni_retiro)&
    ~df_lista_pd['cl_telf1'].isin(cel_retiro)&
    ~df_lista_pd['DNI'].isin(dni_desembolso)
    ].copy()
    
df_lista_pd.shape



(23521, 45)

In [30]:
query = f"""
	SELECT dni_cliente as DNI,date(fecha_envio) as fecha_envio FROM Alice.prospectos_correos_alfin 
    where estado='enviado'
    and fecha_envio>='2026-08-01'
"""
df_correo_1 = pd.read_sql(query, engine_mysql)

In [31]:
df_correo_1["q_envio"] = (
    df_correo_1.groupby("DNI")["DNI"]
    .transform("count")
)

df_correo_1["cantidad_repeticiones"] = (
    df_correo_1
    .groupby("DNI")["DNI"]
    .transform("count")
)

df_correo_1["fecha_envio"] = pd.to_datetime(
    df_correo_1["fecha_envio"],
    errors="coerce"
)
df_correo_1 = df_correo_1.sort_values(
    "fecha_envio",
    ascending=False
)
df_correo_1 = (
    df_correo_1
    .drop_duplicates(
        subset="DNI",
        keep="first"
    )
    .reset_index(drop=True)
)

import pandas as pd

# Convertir a fecha
df_correo_1["fecha_envio"] = pd.to_datetime(
    df_correo_1["fecha_envio"],
    errors="coerce"
)

# Cantidad de días desde fecha_envio hasta hoy
df_correo_1["q_dias"] = (
    pd.Timestamp.today().normalize()
    - df_correo_1["fecha_envio"].dt.normalize()
).dt.days

df_correo_1["fecha_envio"] = pd.to_datetime(
    df_correo_1["fecha_envio"],
    errors="coerce"
)



In [ ]:
condicion = (
    (
        (df_correo_1["q_envio"] > 1)
        &
        (df_correo_1["fecha_envio"].dt.normalize().isin([
            "2026-08-20"
        ]))
    )
    |
    (
        (df_correo_1["q_envio"] == 1)
        &
        (df_correo_1["fecha_envio"].dt.normalize() >= "2026-08-17")
    )
)

df_correo_1 = df_correo_1[condicion].copy()

In [32]:
print(df_lista_pd.columns)
print(df_correo_1.columns)


Index(['DNI', 'Agencia', 'TASA_NUEVA', 'Agencia_comercial', 'cl_telf1',
       'COLOR_FINAL', 'COD_USER_V3', 'USER_V3', 'PERFIL_RO', 'campaña',
       'OFERTA_MAX', 'PLAZO', 'CAPACIDAD_MAX', 'FRESCURA', 'rango_deuda',
       'numentidades', 'TOTAL_A_LIQUIDAR', 'TASA_CREDITO_ANTERIOR', 'TASA_1',
       'TASA_2', 'TASA_3', 'TASA_4', 'TASA_5', 'TASA_6', 'TASA_7', 'MGNEG',
       'MARCA_PD', 'AUTORIZACION_DATOS', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA',
       'TIPO_BASE', 'PROPENSION_DISTRIBUCION', 'OFERTA_SS', 'TASA_1_SS',
       'TASA_2_SS', 'TASA_3_SS', 'TASA_4_SS', 'TASA_5_SS', 'TASA_6_SS',
       'TASA_7_SS', 'ALERTA_MAQUETA', 'FEN', 'PERFIL_ESPECIAL', 'TIPO',
       'tipo_archivo'],
      dtype='object')
Index(['DNI', 'fecha_envio', 'q_envio', 'cantidad_repeticiones', 'q_dias'], dtype='object')


In [33]:
df_lista_pd_1=df_lista_pd.merge(df_correo_1,on='DNI',how='left')

In [ ]:
df_lista_pd_1

In [34]:

ruta_archivo = os.path.join(ruta_csv, 'queda_alfin.csv')
df_lista_pd_1.to_csv(ruta_archivo, sep=';')

In [147]:

def completar_dni(df):
    return df.withColumn(
        "dni_cliente",
        F.lpad(F.col("dni_cliente").cast("string"), 8, "0")
    )

df_formato = completar_dni(df_formato)
df_lista = completar_dni(df_lista)

In [148]:
df_lista=df_lista.select(F.col('dni_cliente').alias('NUMERO_DOCUMENTO'), F.col('nombre_cliente').alias('NOMBRES'),F.col('celular').alias('cl_telf1'), F.col('agencia_atencion').alias('Agencia_comercial'))

In [149]:
print(df_validar.columns)
print(df_base.columns)

['DNI', 'COLOR_FINAL', 'COD_USER_V3', 'USER_V3', 'PERFIL_RO', 'campaña', 'OFERTA_MAX', 'PLAZO', 'CAPACIDAD_MAX', 'FRESCURA', 'rango_deuda', 'numentidades', 'TOTAL_A_LIQUIDAR', 'TASA_CREDITO_ANTERIOR', 'TASA_1', 'TASA_2', 'TASA_3', 'TASA_4', 'TASA_5', 'TASA_6', 'TASA_7', 'MGNEG', 'MARCA_PD', 'AUTORIZACION_DATOS', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'TIPO_BASE', 'PROPENSION_DISTRIBUCION', 'OFERTA_SS', 'TASA_1_SS', 'TASA_2_SS', 'TASA_3_SS', 'TASA_4_SS', 'TASA_5_SS', 'TASA_6_SS', 'TASA_7_SS', 'ALERTA_MAQUETA', 'FEN', 'PERFIL_ESPECIAL', 'TIPO', 'tipo_archivo']
['TIPO_DOI', 'NUMERO_DOCUMENTO', 'NOMBRES', 'APELLIDO_PATERNO', 'APELLIDO_MATERNO', 'SUCURSAL', 'TIENDA', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'FEC_NACIMIENTO', 'OFERTA_MAX', 'OFERTA_REEN', 'Tipo_verificacion', 'GRUPO_RIESGO', 'proveedor', 'lote', 'RETIRO', 'Tasa_1', 'Tasa_2', 'Tasa_3', 'Tasa_4', 'Tasa_5', 'Tasa_6', 'Tasa_7', 'segmento', 'Campana', 'PLAZO', 'TEM', 'PROPENSION_IC', 'Desgravamen', 'CUOTA', 'Edad', 'Oferta_12M', 'Ta

In [150]:
df_validar=df_validar.select(
 F.col('DNI').alias('NUMERO_DOCUMENTO'),
 F.col('COLOR_FINAL').alias('color_final'),
 'USER_V3',
 'PERFIL_RO',
 'OFERTA_MAX',
 'PLAZO',
 'CAPACIDAD_MAX',
 'FRESCURA',
 F.col('campaña').alias('campania'),
 F.col('TASA_1').alias('Tasa_1'),
 F.col('TASA_2').alias('Tasa_2'),
 F.col('TASA_3').alias('Tasa_3'),
 F.col('TASA_4').alias('Tasa_4'),
 F.col('TASA_5').alias('Tasa_5'),
 F.col('TASA_6').alias('Tasa_6'),
 F.col('TASA_7').alias('Tasa_7'),
 'MGNEG',
 'GRUPO_TASA',
 'TIPO_BASE',
 F.col('PROPENSION_DISTRIBUCION').alias('PROPENSION_IC'),
 F.lit('INVENTARIO').alias('lote'),
 F.lit('CET').alias('CRUCE')
)

In [114]:
vali=set(df_validar_2.columns)
bas=set(df_base.columns)

In [ ]:
print(bas-vali)
print(vali-bas)
print(bas & vali)



{'RETIRO_DESEMBOLSO', 'NUEVOS_6M', 'FLG_DEUDA_PLUS', 'MES_GESTION', 'CUOTA_18M', 'CUOTA_24M', 'FLAT2', 'segmento', 'cl_telf9', 'cl_telf8', 'cl_movil', 'GARANTIA', 'Entidad_3', 'fecha_alimentacion', 'cl_fecha_ant', 'cl_hora_gestion', 'RESULTADO', 'Deuda_2', 'COD_BD', 'CAMP_BONO', 'PROMOCION', 'Entidad_1', 'Campana', 'TEM', 'cl_telf7', 'incremento_monto_riesgos', 'tipo_cliente_riegos', 'PROPENSION', 'APELLIDO_PATERNO', 'PERIODO', 'cl_telefono', 'NUEVOS_3M', 'DESEMBOLSADO', 'OFERTA_REEN', 'NOMB_BD', 'cl_celular', 'cl_orden', 'NUM_ENRIQUECIDO', 'Desgravamen_24M', 'color', 'Desgravamen_12M', 'Entidad_2', 'Edad', 'cl_turno', 'NUEVOS_4M', 'CUOTA', 'USUARIO', 'Fecha_Envio', 'RETIRO_GEST', 'marca3', 'Agencia_comercial', 'FECHA_SOL', 'PILOTO_PLAZAS', 'SERVICIO', 'Oferta_24M', 'cl_predictivo', 'cl_telf4', 'Oferta_12M', 'GRUPO_RIESGO', 'FLG_AAHH', 'Tipo_verificacion', 'FLAG_REENG', 'marca2', 'STATUS', 'FEC_NACIMIENTO', 'ACCION', 'cl_fecha_llamar', 'cl_telf5', 'cl_prioridad', 'REP2', 'cl_telf2', 'O

In [85]:
print([row['MARCA_PD' ] for row in df_validar_2.select('MARCA_PD').distinct().collect()])


['A2', 'A4', 'A3', 'A1']


In [78]:
df_base.show(5)

+--------+----------------+--------------+----------------+----------------+--------+------+------------+----------------+--------+--------------+----------+-----------+-----------------+------------+---------+----------+------+------+------+------+------+------+------+------+--------+-------+-----+----+-------------+-----------+-------+----+----------+--------+---------------+---------+----------+--------+---------------+---------+----------+--------+---------------+---------+----------+--------+---------------+---------+------------------+---------+----------------+-------+---------+-------+---------+-------+---------+------------------+-----------------+--------------------+---------+---------------------+-----+---------------+----------+------------+--------+-----------------------+------------+------------+-------------+----+----------+---------+-------------+-------------+---------+---------+---------+----------+---------+---------------+-------------+-------+--------------------

In [151]:

df_lista=df_lista.join(df_base.select('NUMERO_DOCUMENTO'),['NUMERO_DOCUMENTO'],'leftanti')
df_lista=df_lista.join(df_validar,['NUMERO_DOCUMENTO'],'inner')
df_lista=df_lista.dropDuplicates(['NUMERO_DOCUMENTO'])


df_lista = df_lista.withColumn("Campana", F.lit("202608"))
df_lista = df_lista.withColumn("lote", F.lit("INVENTARIO"))
df_lista = df_lista.withColumn("CRUCE", F.lit("CET"))
df_lista = df_lista.withColumn("cl_base", F.lit("Agosto 2026"))
df_lista = df_lista.withColumn("cl_carga", F.lit('2026-08-01'))
df_lista.count()


7298

In [152]:
for i, col in enumerate(df_lista.columns):
    print(i, col)

0 NUMERO_DOCUMENTO
1 NOMBRES
2 cl_telf1
3 Agencia_comercial
4 color_final
5 USER_V3
6 PERFIL_RO
7 OFERTA_MAX
8 PLAZO
9 CAPACIDAD_MAX
10 FRESCURA
11 campania
12 Tasa_1
13 Tasa_2
14 Tasa_3
15 Tasa_4
16 Tasa_5
17 Tasa_6
18 Tasa_7
19 MGNEG
20 GRUPO_TASA
21 TIPO_BASE
22 PROPENSION_IC
23 lote
24 CRUCE
25 Campana
26 cl_base
27 cl_carga


In [ ]:
['NUMERO_DOCUMENTO', 'NOMBRES', 'cl_telf1', 'Agencia_comercial', 'TIPO_DOI', 'NOMBRES', 'APELLIDO_PATERNO', 'APELLIDO_MATERNO', 'SUCURSAL', 'TIENDA', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'FEC_NACIMIENTO', 'OFERTA_MAX', 'OFERTA_REEN', 'Tipo_verificacion', 'GRUPO_RIESGO', 'proveedor', 'lote', 'RETIRO', 'Tasa_1', 'Tasa_2', 'Tasa_3', 'Tasa_4', 'Tasa_5', 'Tasa_6', 'Tasa_7', 'segmento', 'Campana', 'PLAZO', 'TEM', 'PROPENSION_IC', 'Desgravamen', 'CUOTA', 'Edad', 'Oferta_12M', 'Tasa_12M', 'Desgravamen_12M', 'CUOTA_12M', 'Oferta_18M', 'Tasa_18M', 'Desgravamen_18M', 'CUOTA_18M', 'Oferta_24M', 'Tasa_24M', 'Desgravamen_24M', 'CUOTA_24M', 'Oferta_36M', 'Tasa_36M', 'Desgravamen_36M', 'CUOTA_36M', 'Validador_Telefono', 'Prioridad', 'Nombre_prioridad', 'Deuda_1', 'Entidad_1', 'Deuda_2', 'Entidad_2', 'Deuda_3', 'Entidad_3', 'sucursal_comercial', 'Agencia_comercial', 'Region_comercial', 'Ubicacion', 'OfertaMaximaSinSeguro', 'color', 'color_final', 'PROPENSION', 'OFERTA_FINAL', 'GARANTIA', 'Oferta_Minima_Paperless', 'RANGO_OFERTA', 'RANGO_SUELDO', 'CAPACIDAD_MAX', 'PEER', 'PROP_COMER', 'TIPO_GEST', 'CLIENTE_NUEVO', 'GRUPO_TASA', 'NUEVOS_3M', 'NUEVOS_6M', 'NUEVOS_9M', 'NUEVOS_12M', 'NUEVOS_4M', 'GRUPO_MONTO', 'TASA_VS_MONTO', 'USUARIO', 'incremento_monto_riesgos', 'FLG_DEUDA_PLUS', 'tipo_cliente_riegos', 'USER_V3', 'LEAD_CALIDAD', 'SEGMENTO_USER', 'RANGO_EDAD', 'RANGO_OFERTA2', 'PERIODO', 'RETIRO_GEST', 'MEJOR_TIPIFICACION', 'STATUS', 'FECHA_SOL', 'BASE', 'RESULTADO', 'NUM_ENRIQUECIDO', 'TIPO_CONTACTO', 'Q_VENTAS', 'LOCALIDAD', 'DESEMBOLSADO', 'MONTO_DESEMBOLSADO', 'SBI', 'CRUCE', 'PREST_PREVIO', 'ID_CLIENTE', 'RANGO_EDAD2', 'Fecha_Envio', 'TIPO_BD', 'COD_BD', 'NOMB_BD', 'MES_GESTION', 'TIPO_CLIENTE', 'GRUPO_TASA_REENGANCHE', 'SALDO_DIFERENCIAL_REENG', 'FLAG_REENG', 'RETIRO_DESEMBOLSO', 'FRESCURA', 'flag_deuda_v_oferta', 'MGNEG', 'PERFIL_RO', 'TIPO_BASE', 'cl_telf1', 'cl_telf2', 'cl_telf3', 'cl_telf4', 'cl_telf5', 'cl_telf6', 'cl_telf7', 'cl_telf8', 'cl_telf9', 'cl_telf10', 'cl_movil', 'cl_celular', 'cl_telefono', 'cl_turno', 'cl_gestor', 'cl_asesor', 'cl_accion', 'cl_gestion', 'SERVICIO', 'cl_fecha_gestion', 'cl_hora_gestion', 'cl_hits', 'cl_fecha_llamar', 'cl_prioridad', 'cl_orden', 'cl_predictivo', 'cl_tiempo', 'cl_base', 'cl_mes', 'cl_carga', 'id_carga', 'cl_area', 'fecha_alimentacion', 'cl_base_ant', 'cl_accion_ant', 'cl_fecha_ant', 'campania', 'PROMOCION', 'PROMOCION2', 'nombre_base', 'NumEntidades', 'p_banco', 'PERFIL_GLOBAL', 'FLG_AAHH', 'SCORE_TELEFONO', 'PILOTO_PLAZAS', 'INTENSIDAD_MAX', 'marca1', 'marca2', 'marca3', 'AÑO_DURACION_BASE', 'MES_DURACION_BASE', 'FLAT2', 'REP1', 'REP2', 'PILOTO_RETENCION', 'CAMP_BONO', 'ACCION']

['NUMERO_DOCUMENTO', 'NOMBRES', 'cl_telf1', 'Agencia_comercial', 'TIPO_DOI', 'NOMBRES', 'APELLIDO_PATERNO', 'APELLIDO_MATERNO', 'SUCURSAL', 'TIENDA', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'FEC_NACIMIENTO', 'OFERTA_MAX', 'OFERTA_REEN', 'Tipo_verificacion', 'GRUPO_RIESGO', 'proveedor', 'lote', 'RETIRO', 'Tasa_1', 'Tasa_2', 'Tasa_3', 'Tasa_4', 'Tasa_5', 'Tasa_6', 'Tasa_7', 'segmento', 'Campana', 'PLAZO', 'TEM', 'PROPENSION_IC', 'Desgravamen', 'CUOTA', 'Edad', 'Oferta_12M', 'Tasa_12M', 'Desgravamen_12M', 'CUOTA_12M', 'Oferta_18M', 'Tasa_18M', 'Desgravamen_18M', 'CUOTA_18M', 'Oferta_24M', 'Tasa_24M', 'Desgravamen_24M', 'CUOTA_24M', 'Oferta_36M', 'Tasa_36M', 'Desgravamen_36M', 'CUOTA_36M', 'Validador_Telefono', 'Prioridad', 'Nombre_prioridad', 'Deuda_1', 'Entidad_1', 'Deuda_2', 'Entidad_2', 'Deuda_3', 'Entidad_3', 'sucursal_comercial', 'Agencia_comercial', 'Region_comercial', 'Ubicacion', 'OfertaMaximaSinSeguro', 'color', 'color_final', 'PROPENSION', 'OFERTA_FINAL', 'GARANTIA', 'Oferta_Mi

In [131]:

df_lista = df_lista.withColumn("Campana", F.lit("202608"))
df_lista = df_lista.withColumn("lote", F.lit("INVENTARIO"))
df_lista = df_lista.withColumn("CRUCE", F.lit("CET"))
df_lista = df_lista.withColumn("cl_base", F.lit("Agosto 2026"))
df_lista = df_lista.withColumn("cl_carga", F.lit('2026-08-01'))

In [ ]:
|

9023

In [134]:
df_lista.show(2)

+----------------+--------------------+---------+-----------------+--------+--------------------+----------------+----------------+--------+------+------------+---------+--------+--------------+----------+-----------+-----------------+------------+---------+----------+------+------+------+------+------+------+------+------+--------+-------+-----+----+-------------+-----------+-----+----+----------+--------+---------------+---------+----------+--------+---------------+---------+----------+--------+---------------+---------+----------+--------+---------------+---------+------------------+---------+----------------+-------+---------+-------+---------+-------+---------+------------------+-----------------+----------------+---------+---------------------+-----+-----------+----------+------------+--------+-----------------------+------------+------------+-------------+----+----------+---------+-------------+-------------+---------+---------+---------+----------+---------+-----------+--------

In [ ]:
aa

In [144]:
for i, col in enumerate(df_lista.columns):
    print(i, col)

0 NUMERO_DOCUMENTO
1 NOMBRES
2 cl_telf1
3 Agencia_comercial
4 TIPO_DOI
5 APELLIDO_PATERNO
6 APELLIDO_MATERNO
7 SUCURSAL
8 TIENDA
9 DEPARTAMENTO
10 PROVINCIA
11 DISTRITO
12 FEC_NACIMIENTO
13 OFERTA_MAX
14 OFERTA_REEN
15 Tipo_verificacion
16 GRUPO_RIESGO
17 proveedor
18 lote
19 RETIRO
20 Tasa_1
21 Tasa_2
22 Tasa_3
23 Tasa_4
24 Tasa_5
25 Tasa_6
26 Tasa_7
27 segmento
28 Campana
29 PLAZO
30 TEM
31 PROPENSION_IC
32 Desgravamen
33 CUOTA
34 Edad
35 Oferta_12M
36 Tasa_12M
37 Desgravamen_12M
38 CUOTA_12M
39 Oferta_18M
40 Tasa_18M
41 Desgravamen_18M
42 CUOTA_18M
43 Oferta_24M
44 Tasa_24M
45 Desgravamen_24M
46 CUOTA_24M
47 Oferta_36M
48 Tasa_36M
49 Desgravamen_36M
50 CUOTA_36M
51 Validador_Telefono
52 Prioridad
53 Nombre_prioridad
54 Deuda_1
55 Entidad_1
56 Deuda_2
57 Entidad_2
58 Deuda_3
59 Entidad_3
60 sucursal_comercial
61 Region_comercial
62 Ubicacion
63 OfertaMaximaSinSeguro
64 color
65 color_final
66 PROPENSION
67 OFERTA_FINAL
68 GARANTIA
69 Oferta_Minima_Paperless
70 RANGO_OFERTA
71 RANGO_S

In [139]:
df_lista.show()

+----------------+--------------------+---------+-----------------+--------+----------------+----------------+--------+------+------------+---------+------------------+--------------+----------+-----------+-----------------+------------+---------+----------+------+------+------+------+------+------+------+------+--------+-------+-----+----+-------------+-----------+------+----+----------+--------+---------------+---------+----------+--------+---------------+---------+----------+--------+---------------+---------+----------+--------+---------------+---------+------------------+---------+----------------+-------+---------+-------+---------+-------+---------+------------------+-----------------+----------------+---------+---------------------+-----+---------------+----------+------------+--------+-----------------------+------------+------------+-------------+----+----------+---------+-------------+-------------+---------+---------+---------+----------+---------+-----------+-------------+

In [142]:
columnas_originales = df_lista.columns

# Renombrar temporalmente por posición
df_temp = df_lista.toDF(
    *[f"col_{i}" for i in range(len(columnas_originales))]
)

# Eliminar la posición 5
df_temp = df_temp.drop("col_61")

# Recuperar los nombres, excepto el eliminado
columnas_finales = [
    c for i, c in enumerate(columnas_originales)
    if i != 61
]

df_lista = df_temp.toDF(*columnas_finales)

In [153]:
# append_table_SQL(spark,df_prueba,f'ups',server_sa,user_sa,pwd_sa,'ODIN')
# append_table_SQL(spark,df_prueba,f'ups',server_zeus,user_zeus,pwd_zeus,'ODIN')
append_table_SQL(spark,df_lista,f'Base_Maestra_Alfin_bk',server_kishin,user_kishin,pwd_kishin,'DANTALION')


In [154]:
spark.stop()

In [ ]:
{'RETIRO_DESEMBOLSO', 'NUEVOS_6M', 'FLG_DEUDA_PLUS', 'MES_GESTION', 'CUOTA_18M', 'CUOTA_24M', 'FLAT2', 'segmento', 'cl_telf9', 'cl_telf8', 'cl_movil', 'GARANTIA', 'Entidad_3', 'fecha_alimentacion', 'cl_fecha_ant', 'cl_hora_gestion', 'RESULTADO', 'Deuda_2', 'COD_BD', 'CAMP_BONO', 'PROMOCION', 'Entidad_1', 'Campana', 'TEM', 'cl_telf7', 'incremento_monto_riesgos', 'tipo_cliente_riegos', 'PROPENSION', 'APELLIDO_PATERNO', 'PERIODO', 'cl_telefono', 'NUEVOS_3M', 'DESEMBOLSADO', 'OFERTA_REEN', 'NOMB_BD', 'cl_celular', 'cl_orden', 'NUM_ENRIQUECIDO', 'Desgravamen_24M', 'color', 'Desgravamen_12M', 'Entidad_2', 'Edad', 'cl_turno', 'NUEVOS_4M', 'CUOTA', 'USUARIO', 'Fecha_Envio', 'RETIRO_GEST', 'marca3', 'Agencia_comercial', 'FECHA_SOL', 'PILOTO_PLAZAS', 'SERVICIO', 'Oferta_24M', 'cl_predictivo', 'cl_telf4', 'Oferta_12M', 'GRUPO_RIESGO', 'FLG_AAHH', 'Tipo_verificacion', 'FLAG_REENG', 'marca2', 'STATUS', 'FEC_NACIMIENTO', 'ACCION', 'cl_fecha_llamar', 'cl_telf5', 'cl_prioridad', 'REP2', 'cl_telf2', 'Oferta_36M', 'Oferta_18M', 'LEAD_CALIDAD', 'SBI', 'cl_fecha_gestion', 'cl_hits', 'OFERTA_FINAL', 'cl_tiempo', 'INTENSIDAD_MAX', 'cl_gestor', 'Desgravamen_36M', 'CLIENTE_NUEVO', 'proveedor', 'NumEntidades', 'TIENDA', 'PREST_PREVIO', 'cl_accion', 'PROP_COMER', 'Desgravamen_18M', 'ID_CLIENTE', 'RANGO_EDAD', 'cl_carga', 'Tasa_24M', 'Prioridad', 'SUCURSAL', 'MONTO_DESEMBOLSADO', 'cl_base', 'p_banco', 'TIPO_GEST', 'GRUPO_MONTO', 'cl_asesor', 'cl_accion_ant', 'REP1', 'Deuda_3', 'RANGO_SUELDO', 'MEJOR_TIPIFICACION', 'TASA_VS_MONTO', 'cl_telf10', 'DISTRITO', 'TIPO_BD', 'LOCALIDAD', 'sucursal_comercial', 'cl_base_ant', 'PILOTO_RETENCION', 'RANGO_OFERTA2', 'nombre_base', 'SALDO_DIFERENCIAL_REENG', 'cl_gestion', 'BASE', 'RANGO_OFERTA', 'PROVINCIA', 'Ubicacion', 'MES_DURACION_BASE', 'CUOTA_36M', 'cl_area', 'Tasa_18M', 'AÑO_DURACION_BASE', 'APELLIDO_MATERNO', 'flag_deuda_v_oferta', 'OfertaMaximaSinSeguro', 'NOMBRES', 'cl_telf1', 'id_carga', 'Tasa_12M', 'PROMOCION2', 'RANGO_EDAD2', 'marca1', 'Tasa_36M', 'NUEVOS_9M', 'NUEVOS_12M', 'Oferta_Minima_Paperless', 'DEPARTAMENTO', 'Desgravamen', 'Nombre_prioridad', 'Deuda_1', 'TIPO_CLIENTE', 'cl_telf6', 'CUOTA_12M', 'PEER', 'SEGMENTO_USER', 'RETIRO', 'Q_VENTAS', 'PERFIL_GLOBAL', 'SCORE_TELEFONO', 'cl_telf3', 'Validador_Telefono', 'TIPO_DOI', 'TIPO_CONTACTO', 'Region_comercial', 'GRUPO_TASA_REENGANCHE', 'cl_mes'}


In [60]:
query = """
select  *  from DANTALION.dbo.Base_Maestra_ALFIN_BK
where Fecha_Envio>='2026-08-01'
    """
df_base=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)

In [ ]:
df_base=df_base.drop('cdv_alfin_banco','monto_solicitado','fecha_visita','ejecutivo_target','numentidades','fecha_visita','estado','monto_solicitado','fecha_registro','cdv_alfin_banco','TASA_2_SS','supervisor','fecha_envio','_c0','fecha_dia','TASA_CREDITO_ANTERIOR','TASA_6_SS','FEN','id','intentos_realizados','TASA_7_SS','canal_campo','hora_visita','OFERTA_SS','TASA_3_SS','TASA_1_SS','correo_envio','AUTORIZACION_DATOS','COD_USER_V3','PERFIL_ESPECIAL','ejecutivo_target','ALERTA_MAQUETA')
base=set(df_base.columns)
lista1=set(df_lista.columns)

AttributeError: 'int' object has no attribute 'withColumn'

In [ ]:
df_base=df_base.withColumnRenamed('')

In [50]:
print(lista1-base)
print(base-lista1)
print(base&lista1)


{'fecha_registro', 'cdv_alfin_banco', 'TASA_2_SS', 'campaña', 'celular', 'supervisor', 'fecha_envio', 'MARCA_PD', 'codigo_ejecutivo_id', '_c0', 'fecha_dia', 'TASA_CREDITO_ANTERIOR', 'TASA_6_SS', 'FEN', 'TASA_3', 'TASA_1', 'TIPO', 'TASA_4', 'TASA_5_SS', 'id', 'TASA_6', 'TASA_4_SS', 'intentos_realizados', 'TASA_7_SS', 'TASA_7', 'canal_campo', 'agencia_atencion', 'hora_visita', 'OFERTA_SS', 'COLOR_FINAL', 'tipo_archivo', 'TASA_3_SS', 'TASA_1_SS', 'correo_envio', 'AUTORIZACION_DATOS', 'COD_USER_V3', 'PERFIL_ESPECIAL', 'rango_deuda', 'estado', 'fecha_visita', 'FLAG_DEUDA_V_OFERTA', 'nombre_cliente', 'tipo_carga', 'TASA_5', 'ALERTA_MAQUETA', 'DNI', 'PROPENSION_DISTRIBUCION', 'TOTAL_A_LIQUIDAR', 'numentidades', 'ejecutivo_target', 'monto_solicitado', 'TASA_2'}
{'RETIRO_DESEMBOLSO', 'NUEVOS_6M', 'FLG_DEUDA_PLUS', 'MES_GESTION', 'CUOTA_18M', 'CUOTA_24M', 'FLAT2', 'segmento', 'cl_telf9', 'cl_telf8', 'cl_movil', 'GARANTIA', 'Entidad_3', 'fecha_alimentacion', 'cl_fecha_ant', 'cl_hora_gestion', 'RE

In [ ]:
{'RETIRO_DESEMBOLSO', 'NUEVOS_6M', 'FLG_DEUDA_PLUS', 'MES_GESTION', 'CUOTA_18M', 'CUOTA_24M', 'FLAT2', 'segmento', 'cl_telf9', 'cl_telf8', 'cl_movil', 'GARANTIA', 'Entidad_3', 'fecha_alimentacion', 'cl_fecha_ant', 'cl_hora_gestion', 'RESULTADO', 'Deuda_2', 'COD_BD', 'CAMP_BONO', 'PROMOCION', 'Entidad_1', 'Tasa_4', 'Tasa_7', 'Campana', 'TEM', 'cl_telf7', 'incremento_monto_riesgos', 'tipo_cliente_riegos', 'PROPENSION', 'APELLIDO_PATERNO', 'PERIODO', 'cl_telefono', 'NUEVOS_3M', 'DESEMBOLSADO', 'OFERTA_REEN', 'NOMB_BD', 'cl_celular', 'cl_orden', 'NUM_ENRIQUECIDO', 'NUMERO_DOCUMENTO', 'Desgravamen_24M', 'Desgravamen_12M', 'Entidad_2', 'Edad', 'cl_turno', 'NUEVOS_4M', 'CUOTA', 'USUARIO', 'RETIRO_GEST', 'marca3', 'Agencia_comercial', 'FECHA_SOL', 'PILOTO_PLAZAS', 'PROPENSION_IC', 'SERVICIO', 'Oferta_24M', 'cl_predictivo', 'cl_telf4', 'Oferta_12M', 'GRUPO_RIESGO', 'marca2', 'FLG_AAHH', 'Tipo_verificacion', 'Tasa_5', 'FLAG_REENG', 'STATUS', 'FEC_NACIMIENTO', 'ACCION', 'cl_fecha_llamar', 'cl_telf5', 'cl_prioridad', 'REP2', 'cl_telf2', 'Oferta_36M', 'Oferta_18M', 'LEAD_CALIDAD', 'SBI', 'campania', 'cl_fecha_gestion', 'cl_hits', 'OFERTA_FINAL', 'cl_tiempo', 'INTENSIDAD_MAX', 'cl_gestor', 'Desgravamen_36M', 'CLIENTE_NUEVO', 'proveedor', 'TIENDA', 'PREST_PREVIO', 'cl_accion', 'PROP_COMER', 'Desgravamen_18M', 'ID_CLIENTE', 'color_final', 'RANGO_EDAD', 'CRUCE', 'cl_carga', 'Tasa_24M', 'Prioridad', 'SUCURSAL', 'MONTO_DESEMBOLSADO', 'cl_base', 'p_banco', 'TIPO_GEST', 'GRUPO_MONTO', 'cl_asesor', 'cl_accion_ant', 'REP1', 'Deuda_3', 'RANGO_SUELDO', 'MEJOR_TIPIFICACION', 'TASA_VS_MONTO', 'cl_telf10', 'DISTRITO', 'TIPO_BD', 'LOCALIDAD', 'sucursal_comercial', 'cl_base_ant', 'PILOTO_RETENCION', 'RANGO_OFERTA2', 'nombre_base', 'SALDO_DIFERENCIAL_REENG', 'cl_gestion', 'BASE', 'RANGO_OFERTA', 'PROVINCIA', 'Tasa_6', 'Ubicacion', 'Tasa_1', 'MES_DURACION_BASE', 'CUOTA_36M', 'cl_area', 'Tasa_18M', 'AÑO_DURACION_BASE', 'APELLIDO_MATERNO', 'flag_deuda_v_oferta', 'OfertaMaximaSinSeguro', 'NOMBRES', 'cl_telf1', 'id_carga', 'Tasa_12M', 'PROMOCION2', 'RANGO_EDAD2', 'marca1', 'Tasa_36M', 'NUEVOS_9M', 'NUEVOS_12M', 'lote', 'Oferta_Minima_Paperless', 'DEPARTAMENTO', 'Desgravamen', 'Nombre_prioridad', 'Deuda_1', 'TIPO_CLIENTE', 'cl_telf6', 'CUOTA_12M', 'PEER', 'SEGMENTO_USER', 'RETIRO', 'Q_VENTAS', 'PERFIL_GLOBAL', 'SCORE_TELEFONO', 'cl_telf3', 'Validador_Telefono', 'TIPO_DOI', 'TIPO_CONTACTO', 'Region_comercial', 'GRUPO_TASA_REENGANCHE', 'cl_mes', 'Tasa_3', 'Tasa_2'}
fecha

In [52]:
df_base.show(2)

+--------+----------------+-------------+----------------+----------------+--------+------+------------+----------------+--------+--------------+----------+-----------+-----------------+------------+---------+----------+------+------+------+------+------+------+------+------+--------+-------+-----+----+-------------+-----------+-------+----+----------+--------+---------------+---------+----------+--------+---------------+---------+----------+--------+---------------+---------+----------+--------+---------------+---------+------------------+---------+----------------+-------+---------+-------+---------+-------+---------+------------------+-----------------+--------------------+---------+---------------------+-----+-------------+----------+------------+--------+-----------------------+------------+------------+-------------+----+----------+---------+-------------+-------------+---------+---------+---------+----------+---------+--------------+-------------+-------+------------------------

In [5]:
df_lista=df_lista.join(df_validar,['DNI'],'inner')
df_lista=df_lista.dropDuplicates(['DNI'])
df_lista.count()

936

In [7]:
df_lista_pd=df_lista.toPandas()

ruta_archivo = os.path.join(ruta_csv, 'aqui.csv')
df_lista_pd.to_csv(ruta_archivo, sep=';')

In [11]:
df_lista=df_lista.select(F.col('dni_cliente').alias('DNI'),'celular')

In [12]:
df_lista=df_lista.join(df_validar,['DNI'],'inner')

In [12]:
query = """
select  NUMERO_DOCUMENTO as DNI from DANTALION.dbo.Base_Maestra_ALFIN_BK
where Fecha_Envio>='2026-08-01'

    """
df_formato=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)

In [13]:

def completar_dni(df):
    return df.withColumn(
        "DNI",
        F.lpad(F.col("DNI").cast("string"), 8, "0")
    )

df_formato = completar_dni(df_formato)
df_validar = completar_dni(df_validar)

In [14]:
df_formato.count()

233296

In [15]:
df_formato=df_formato.join(df_validar,['DNI'],'inner')

In [17]:
df_formatod=df_formato.dropDuplicates(['DNI'])
df_formatod.count()


233296

In [18]:
append_table_SQL(spark,df_formatod,f'borrar_aaa',server_kishin,user_kishin,pwd_kishin,'DANTALION')


In [4]:
df_lista=df_lista.withColumn('hoja',F.lit(0))

In [5]:
df_lista=df_lista.withColumnRenamed('nombres','nombre')
df_lista=df_lista.withColumnRenamed('DNI','dni')

In [6]:
df_lista = completar_con_ceros(df_lista, "DNI")
df_lista_en_uso = completar_con_ceros(df_lista_en_uso, "DNI")
df_validar = completar_con_ceros(df_validar, "DNI")

In [7]:
df_lista=df_lista.select('dni', 'nombre', 'celular', 'agencia', 'hoja')
df_lista_en_uso=df_lista_en_uso.select('dni', 'nombre', 'celular', 'agencia', 'hoja')

In [8]:
df_lista=df_lista.unionByName(df_lista_en_uso)

In [ ]:
df_lista_1=df_lista.join(df_validar,['DNI'],'inner')
# df_lista_1=df_lista.join(df_lista_en_uso,['DNI'],'left')
df_lista_1=df_lista_1.dropDuplicates(['DNI'])
print(df_lista_1.columns)


['dni', 'nombre', 'celular', 'agencia', 'hoja', 'COLOR_FINAL', 'COD_USER_V3', 'USER_V3', 'PERFIL_RO', 'campaña', 'OFERTA_MAX', 'PLAZO', 'CAPACIDAD_MAX', 'FRESCURA', 'rango_deuda', 'numentidades', 'TOTAL_A_LIQUIDAR', 'TASA_CREDITO_ANTERIOR', 'TASA_1', 'TASA_2', 'TASA_3', 'TASA_4', 'TASA_5', 'TASA_6', 'TASA_7', 'MGNEG', 'MARCA_PD', 'AUTORIZACION_DATOS', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'TIPO_BASE', 'PROPENSION_DISTRIBUCION', 'OFERTA_SS', 'TASA_1_SS', 'TASA_2_SS', 'TASA_3_SS', 'TASA_4_SS', 'TASA_5_SS', 'TASA_6_SS', 'TASA_7_SS', 'ALERTA_MAQUETA', 'FEN', 'PERFIL_ESPECIAL', 'TIPO', 'tipo_archivo']


In [13]:
df_lista=df_lista.dropDuplicates(['DNI'])
df_lista_pd=df_lista.toPandas()

In [14]:
filename='TARGET.txt'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_target_desembolso = pd.read_csv(ruta_archivo,sep='|')
df_target_desembolso = df_target_desembolso[['DNI']].copy()
df_target_desembolso['CANAL']='CANAL'
filename='ACUM_DESEM.txt'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_fugas = pd.read_csv(ruta_archivo,sep='|')
df_fugas = df_fugas[['DNI','CANALVENTA']].copy()
df_target_desembolso['DNI'] = (
    df_target_desembolso['DNI']
    .astype(str)
    .str.replace(r'\D', '', regex=True)   
    .replace('', pd.NA)                     
    .str.zfill(8)                           
)
df_fugas['DNI'] = (
    df_fugas['DNI']
    .astype(str)
    .str.replace(r'\D', '', regex=True)   
    .replace('', pd.NA)                     
    .str.zfill(8)                           
)

df_desembolso=df_fugas.merge(
    df_target_desembolso,
    on=['DNI'],
    how='left'
)
df_desembolso = df_desembolso.fillna("OTROS")
df_desembolso.rename(columns={'DNI': 'dni_cliente'}, inplace=True)


filename='RetiroDefinitivo_BlackList.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_def_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_BlackList.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_Telefonos.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_telf = pd.read_csv(ruta_archivo,sep='|')
filename='retiro_correo_alfin.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_retiro_correo = pd.read_csv(ruta_archivo,sep=';')

# filename='desembolso.csv'
# ruta_archivo = os.path.join(ruta_alfin, filename)
# df_des = pd.read_csv(ruta_archivo,sep=';')

df_def_blacklist = df_def_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_blacklist = df_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_telf= df_telf.rename(columns={'TELEFONO': 'celular'})
df_retiro_correo= df_retiro_correo.rename(columns={'DNI': 'dni_cliente'})

# Blacklists de DNI
# df6 = df_des.copy()
# df6["celular"] = None
# df6 = df6[["dni_cliente", "celular"]]

# Blacklists de DNI
df1 = df_def_blacklist.copy()
df1["celular"] = None
df1 = df1[["dni_cliente", "celular"]]

df2 = df_blacklist.copy()
df2["celular"] = None
df2 = df2[["dni_cliente", "celular"]]

# Blacklist de teléfonos
df3 = df_telf.copy()
df3["dni_cliente"] = None
df3 = df3[["dni_cliente", "celular"]]

# Archivo con DNI y celular
df4 = df_retiro_correo[["dni_cliente", "celular"]].copy()

# Unir todo
df_retiros = pd.concat(
    [df1, df2, df3, df4],
    ignore_index=True
)

dni_retiro = set(df_retiros['dni_cliente'].dropna())
cel_retiro = set(df_retiros['celular'].dropna())
dni_desembolso = set(df_desembolso['dni_cliente'].dropna())





C:\Users\DATA\AppData\Local\Temp\ipykernel_17800\2722451366.py:79: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_retiros = pd.concat(


In [15]:


df_lista_pd=df_lista_pd[
    ~df_lista_pd['DNI'].isin(dni_retiro)&
    ~df_lista_pd['celular'].isin(cel_retiro)&
    ~df_lista_pd['DNI'].isin(dni_desembolso)
    ].copy()
df_lista_pd.shape


(4293, 42)

In [11]:
from pyspark.sql.window import Window

# Ventana por dni ordenando "uno" de mayor a menor
w = Window.partitionBy("dni").orderBy(F.col("hoja").asc())

df_lista_2 = (
    df_lista_1
    .withColumn(
        "rn",
        F.row_number().over(w)
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
)

In [12]:
# df_lista_pd=df_lista_1.toPandas()
df_lista_pd_v2=df_lista_2.toPandas()

In [13]:
server_sql = server_zeus
db_sql = "THOTH"
user_sql = user_zeus
pwd_sql = pwd_zeus

engine = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
query = f"""
	SELECT distinct Dni,Descripcion_ FROM THOTH.dbo.Tmp_LLamadas_Alfin 
    where Descripcion_ in(
    'TELEFONO FUERA DE SERVICIO / NO EXISTE',
       'EXPRESO RECIBIR MÚLTIPLES LLAMADAS',
       'SOLICITÓ NO SER CONTACTADO',
       'FUERA DE SERVICIO',
       'EXPRESO FUTURA DENUNCIA ANTE INDECOPI O REGULADOR',
       'EXPRESO QUE NO AUTORIZÓ USO DE DATOS PERSONALES'
    )
"""
df_tipis = pd.read_sql(query, engine)

set_tipi = set(
    df_tipis['Dni']
    .dropna()
    .drop_duplicates()
)

In [15]:
df_lista_pd_v2.head()

,dni,nombre,celular,agencia,hoja,COLOR_FINAL,COD_USER_V3,USER_V3,PERFIL_RO,campaña,...,TASA_3_SS,TASA_4_SS,TASA_5_SS,TASA_6_SS,TASA_7_SS,ALERTA_MAQUETA,FEN,PERFIL_ESPECIAL,TIPO,tipo_archivo
0,00002331,ISUIZA USHIÃ‘AHUA OTTO,992027660,PUCALLPA,2,AMARILLO OSCURO,7,7. Peers,Aceptante II,SOLODNI,...,104,104,104,104,104,None,None,None,PRESTALTOKE,campo
1,00003012,MELCHOR ANTONIO MERCADO RUIZ,950582801,PUCALLPA,1,NARANJA OSCURO,3,3. MES + PLD Peers,Aceptante I,SOLODNI,...,110,110,110,110,110,None,None,None,PRESTALTOKE,campo
2,00003830,ROSA ISUIZA SINARAHUA,961925976,PUCALLPA,3,VERDE CLARO,7,7. Peers,Aceptante I,MUJER,...,70,65,62,56,56,None,None,None,PRESTALTOKE,campo
3,00008040,MENESES GAVILAN SAUL ENRIQUE,961511952,PUCALLPA,2,VERDE OSCURO,1,1. sunedu & sunarp A,Aceptante I,SOLODNI,...,81,72,72,72,72,None,None,None,PRESTALTOKE,campo
4,00011581,ROGER PEREZ VILLALBA,949637983,PUCALLPA,3,AMARILLO CLARO,7,7. Peers,Aceptante I,SDFCP100,...,87,78,78,78,78,None,None,None,PRESTALTOKE,campo


In [16]:
df_lista_pd_v2 = df_lista_pd_v2[
    (~df_lista_pd_v2["dni"].isin(dni_retiro)) &
    (~df_lista_pd_v2["dni"].isin(dni_desembolso)) &
    (~df_lista_pd_v2["dni"].isin(set_tipi)) &
    (~df_lista_pd_v2["celular"].isin(cel_retiro))
].copy()

In [17]:
query = f"""
	SELECT distinct dni_cliente
	FROM Alice.prospectos_correos_alfin
	where fecha_registro>='2026-08-01'
"""
df_prospectos_correos_alfin = pd.read_sql(query, engine_mysql)

In [26]:
query = f"""
	select * from  Alice.agencias_alfin
"""
df_prospectos_correos_alfin = pd.read_sql(query, engine_mysql)
ruta_archivo = os.path.join(ruta_alfin, 'ssss.csv')
df_prospectos_correos_alfin.to_csv(ruta_archivo, sep=';')

In [19]:
set_dni = set(df_prospectos_correos_alfin["dni_cliente"].dropna().unique())

df_lista_pd_v2.loc[
    df_lista_pd_v2["dni"].isin(set_dni),
    "uso"
] = "cargado"

# Completar nulos
df_lista_pd_v2["uso"] = df_lista_pd_v2["uso"].fillna("no")

In [20]:
df_lista_pd_v2["OFERTA_MAX"] = pd.to_numeric(
    df_lista_pd_v2["OFERTA_MAX"],
    errors="coerce"
)

cantidad = df_lista_pd_v2[
                            (df_lista_pd_v2["hoja"].isin(['0'])) |
                            (
                                (df_lista_pd_v2["PROPENSION_DISTRIBUCION"].isin(['1'])) &
                                (df_lista_pd_v2["COLOR_FINAL"].isin([ "VERDE OSCURO"])) &
                                (df_lista_pd_v2["USER_V3"].isin(["7. Peers"])) &
                                (df_lista_pd_v2["PERFIL_RO"].isin(["Aceptante I",'Aceptante II','Dispuesto I'])) &
                                (df_lista_pd_v2["OFERTA_MAX"] >= 10500)
                            )
                    ].shape[0]

print(cantidad)

1396


In [54]:
df_todo = df_lista_pd_v2[
                            (df_lista_pd_v2["hoja"].isin(['0'])) |
                            (
                                (df_lista_pd_v2["PROPENSION_DISTRIBUCION"].isin(['1'])) &
                                (df_lista_pd_v2["COLOR_FINAL"].isin([ "VERDE OSCURO"])) &
                                (df_lista_pd_v2["USER_V3"].isin(["7. Peers"])) &
                                (df_lista_pd_v2["PERFIL_RO"].isin(["Aceptante I",'Aceptante II','Dispuesto I'])) &
                                (df_lista_pd_v2["OFERTA_MAX"] >= 10500)
                            )
                    ].copy()

In [23]:
df_lista_pd_v2=df_lista_pd_v2.dropDuplicate('dni')

AttributeError: 'DataFrame' object has no attribute 'dropDuplicate'

In [16]:

ruta_archivo = os.path.join(ruta_csv, 'alfin_ult_2.csv')
df_lista_pd.to_csv(ruta_archivo, sep=';')

In [103]:
df_todo.columns

Index(['dni_cliente', 'nombre', 'celular', 'agencia_atencion', 'hoja',
       'COLOR_FINAL', 'COD_USER_V3', 'USER_V3', 'PERFIL_RO', 'campaña',
       'monto_solicitado', 'PLAZO', 'CAPACIDAD_MAX', 'FRESCURA', 'rango_deuda',
       'numentidades', 'TOTAL_A_LIQUIDAR', 'TASA_CREDITO_ANTERIOR', 'TASA_1',
       'TASA_2', 'TASA_3', 'TASA_4', 'TASA_5', 'TASA_6', 'TASA_7', 'MGNEG',
       'MARCA_PD', 'AUTORIZACION_DATOS', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA',
       'TIPO_BASE', 'PROPENSION_DISTRIBUCION', 'OFERTA_SS', 'TASA_1_SS',
       'TASA_2_SS', 'TASA_3_SS', 'TASA_4_SS', 'TASA_5_SS', 'TASA_6_SS',
       'TASA_7_SS', 'ALERTA_MAQUETA', 'FEN', 'PERFIL_ESPECIAL', 'TIPO',
       'tipo_archivo', 'uso', 'supervisor', 'canal_campo',
       'codigo_ejecutivo_id', 'ejecutivo_target', 'cdv_alfin_banco',
       'hora_visita', 'telefono_cliente', 'dni_vendedor'],
      dtype='object')

In [139]:

df_todo['supervisor']='CARLOS ENRIQUE RAMIREZ CACHIQUE'
df_todo['canal_campo']='CALL CENTER / TARGET OUTSOURCING'
df_todo['codigo_ejecutivo_id']='00000001'
df_todo['ejecutivo_target']='BOT'
df_todo['cdv_alfin_banco']='ROSA HONOR'

df_todo = df_todo.rename(columns={
    'OFERTA_MAX': 'monto_solicitado',
    'nombre': 'nombre_cliente',
    'agencia': 'agencia_atencion',
    'COLOR_FINAL': 'color',
    'DNI': 'dni_cliente'
})
df_todo["dni_cliente"] = (
    df_todo["dni_cliente"]
    .astype(str)
    .str.zfill(8)
)
df_todo['tipo_gestion']='Derivacion'
df_todo['operador']='TARGET'

df_todo["agencia_atencion"] = (
    df_todo["agencia_atencion"]
    .str.strip()
)
# Horas posibles: 09 a 18
horas = np.random.randint(9, 19, size=len(df_todo))

# Minutos posibles
minutos = np.random.choice([0, 15, 30, 45], size=len(df_todo))

# Crear la columna
df_todo["hora_visita"] = [
    f"{h:02d}:{m:02d}:00"
    for h, m in zip(horas, minutos)
]
df_todo['telefono_cliente']=df_todo['celular']
df_todo['dni_vendedor']=df_todo['ejecutivo_target']


In [140]:
equivalencias = {
    'SAN JUAN DE LURIG': 'SAN JUAN DE LURIGANCHO',
    'SAN JUAN DE LURIG': 'SAN JUAN DE LURIGANCHO',
    'ENMANCIPACION': 'EMANCIPACION',
    'PC HUANCAYO': 'HUANCAYO',
    'PC TACNA': 'TACNA',
    'PC HUARAZ': 'HUARAZ',
    'TRUJ CENTRO': 'TRUJILLO CENTRO',
    'TRUJ AMERICA': 'TRUJILLO AMERICA',
    'AREQ CAYMA': 'AREQUIPA CAYMA',
    'AREQ PAMPILLA': 'AREQUIPA PAMPILLA'
}

df_todo['agencia_atencion'] = (
    df_todo['agencia_atencion']
    .replace(equivalencias)
)
fechas = pd.to_datetime([
    "2026-08-12",
    "2026-08-13",
    "2026-08-14",
    "2026-08-15"
])

df_todo["fecha_visita"] = np.random.choice(
    fechas,
    size=len(df_todo)
)


In [141]:
query = f"""
	select agencia_Formulario as agencia_tienda,agencia_correo as agencia_atencion from Alice.agencias_alfin
"""
df_agencia = pd.read_sql(query, engine_mysql)
df_agencia.head()

,agencia_tienda,agencia_atencion
0,738363 - CAJAMARCA,CAJAMARCA
1,737490 - CASTILLA,CASTILLA
2,734281 - CHICLAYO BALTA,CHICLAYO BALTA
3,734272 - CHIMBOTE,CHIMBOTE
4,738360 - MOSHOQUEQUE,MOSHOQUEQUE


In [142]:
df_ultimo = df_todo.merge(
    df_agencia,
    on="agencia_atencion",
    how="left"
)
df_ultimo[
    df_ultimo["agencia_tienda"].isnull()
][['agencia_atencion','agencia_tienda']].head(10)

,agencia_atencion,agencia_tienda


In [143]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)


In [144]:
df_correo=df_ultimo[['canal_campo', 'supervisor', 'ejecutivo_target', 'codigo_ejecutivo_id', 'cdv_alfin_banco', 'dni_cliente', 'nombre_cliente', 'monto_solicitado', 'celular', 'agencia_atencion', 'fecha_visita','hora_visita','color']] .copy()
df_correo['tipo_carga']='MANUAL'
  
df_formulario=df_ultimo[['dni_vendedor', 'operador', 'dni_cliente', 'nombre_cliente', 'telefono_cliente', 'agencia_tienda', 'fecha_visita', 'monto_solicitado', 'tipo_gestion']].copy()

display(df_correo.head(2))
display(df_formulario.head(2))


,canal_campo,supervisor,ejecutivo_target,codigo_ejecutivo_id,cdv_alfin_banco,dni_cliente,nombre_cliente,monto_solicitado,celular,agencia_atencion,fecha_visita,hora_visita,color,tipo_carga
0,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,00122132,IVETH SILVA VELA,12800.0,975240662,PUCALLPA,2026-08-14,17:45:00,VERDE OSCURO,MANUAL
1,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,00186701,MARIA ELENA ARBILDO CHUMACERO,22000.0,927599160,PUENTE PIEDRA,2026-08-13,16:30:00,VERDE OSCURO,MANUAL


,dni_vendedor,operador,dni_cliente,nombre_cliente,telefono_cliente,agencia_tienda,fecha_visita,monto_solicitado,tipo_gestion
0,BOT,TARGET,00122132,IVETH SILVA VELA,975240662,738334 - PUCALLPA,2026-08-14,12800.0,Derivacion
1,BOT,TARGET,00186701,MARIA ELENA ARBILDO CHUMACERO,927599160,738369 - PUENTE PIEDRA,2026-08-13,22000.0,Derivacion


In [145]:
df_todo.head(3)

,dni_cliente,nombre_cliente,celular,agencia_atencion,hoja,color,COD_USER_V3,USER_V3,PERFIL_RO,campaña,monto_solicitado,PLAZO,CAPACIDAD_MAX,FRESCURA,rango_deuda,numentidades,TOTAL_A_LIQUIDAR,TASA_CREDITO_ANTERIOR,TASA_1,TASA_2,TASA_3,TASA_4,TASA_5,TASA_6,TASA_7,MGNEG,MARCA_PD,AUTORIZACION_DATOS,FLAG_DEUDA_V_OFERTA,GRUPO_TASA,TIPO_BASE,PROPENSION_DISTRIBUCION,OFERTA_SS,TASA_1_SS,TASA_2_SS,TASA_3_SS,TASA_4_SS,TASA_5_SS,TASA_6_SS,TASA_7_SS,ALERTA_MAQUETA,FEN,PERFIL_ESPECIAL,TIPO,tipo_archivo,uso,supervisor,canal_campo,codigo_ejecutivo_id,ejecutivo_target,cdv_alfin_banco,hora_visita,telefono_cliente,dni_vendedor,fecha_visita,operador,tipo_gestion
37,00122132,IVETH SILVA VELA,975240662,PUCALLPA,3,VERDE OSCURO,7,7. Peers,Aceptante I,MUJER,12800.0,36,717.167,4,4,5,None,None,56,55,54,52,48,None,None,None,A1,None,0,Menor tasa,Regular,1,11800,110,93,81,72,72,72,72,2,None,None,PRESTALTOKE,campo,no,CARLOS ENRIQUE RAMIREZ CACHIQUE,CALL CENTER / TARGET OUTSOURCING,00000001,BOT,ROSA HONOR,17:45:00,975240662,BOT,2026-08-14,TARGET,Derivacion
45,00186701,MARIA ELENA ARBILDO CHUMACERO,927599160,PUENTE PIEDRA,1,VERDE OSCURO,7,7. Peers,Aceptante II,MUJER,22000.0,36,2075.78,4,4,3,None,None,56,55,54,52,51,46,None,None,A1,None,1,Menor tasa,SALDO COMPETIDOR,1,22000,110,94,82,74,72,65,65,None,None,None,PRESTALTOKE,campo,cargado,CARLOS ENRIQUE RAMIREZ CACHIQUE,CALL CENTER / TARGET OUTSOURCING,00000001,BOT,ROSA HONOR,16:30:00,927599160,BOT,2026-08-13,TARGET,Derivacion
67,00240847,PATRICIA ROSSANNA YARANGA VITE,974068504,TARAPOTO,3,VERDE OSCURO,7,7. Peers,Aceptante II,REENG,25300.0,48,1198.59,0,3,6,4364.42,54,44,44,44,44,43,42,42,None,A2,S,1,Menor tasa,SALDO COMPETIDOR,1,25300,71,62,59,57,55,52,52,None,None,None,PRESTALTOKE,campo,no,CARLOS ENRIQUE RAMIREZ CACHIQUE,CALL CENTER / TARGET OUTSOURCING,00000001,BOT,ROSA HONOR,18:30:00,974068504,BOT,2026-08-13,TARGET,Derivacion


In [146]:

df_correo.to_sql(
    name="prospectos_correos_alfin",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

df_formulario.to_sql(
    name="prospectos_envio_alfin",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

1446

In [ ]:


df_todo['operador']='TARGET'
df_todo['tipo_gestion']='Derivacion'

fechas = pd.date_range("2026-08-12", "2026-08-13")

df_todo["fecha_visita"] = np.random.choice(fechas, size=len(df_todo))

# Horas posibles: 09 a 18
horas = np.random.randint(9, 19, size=len(df_correo))

# Minutos posibles
minutos = np.random.choice([0, 15, 30, 45], size=len(df_todo))

# Crear la columna
df_todo["hora_visita"] = [
    f"{h:02d}:{m:02d}:00"
    for h, m in zip(horas, minutos)
]

KeyError: 'cod_agencia'

In [ ]:
query = f"""
	SELECT distinct dni_cliente
	FROM Alice.prospectos_correos_alfin
	where fecha_registro>='2026-08-01'
"""
df_prospectos_correos_alfin = pd.read_sql(query, engine_mysql)
query = f"""
	SELECT distinct dni_cliente
	FROM Alice.prospectos_correos_alfin
	where fecha_registro>='2026-08-01'
"""
df_prospectos_correos_alfin = pd.read_sql(query, engine_mysql)

In [125]:

df_lista_pd_v2.loc[
    (df_lista_pd_v2["dni"].isin(dni_desembolso)),
    "uso"
] = "desembolso"

In [126]:

df_lista_pd_v2.loc[
    (df_lista_pd_v2["dni"].isin(dni_retiro)) |
    (df_lista_pd_v2["celular"].isin(cel_retiro)),
    "uso"
] = "retiro"

In [74]:
df_lista_pd_v2['estado'].unique()

array([nan, 'cargado'], dtype=object)

In [127]:
df_lista_pd_v2.loc[
    df_lista_pd_v2["hoja"] == 0,
    "uso"
] = "pendiente"

In [128]:
df_resumen = (
    df_lista_pd_v2
    .groupby("uso")
    .size()
    .reset_index(name="cantidad")
)

print(df_resumen)

          uso  cantidad
0     cargado     11993
1  desembolso       350
2          no     17036
3   pendiente        64
4      retiro         1


In [129]:

ruta_archivo = os.path.join(ruta_alfin, 'validar.csv')
df_lista_pd_v2.to_csv(ruta_archivo, sep=';')

In [ ]:
df_lista_pd=df_lista_pddf_lista_pd

In [ ]:
df_lista_pd

In [6]:
df_lista_pd["agencia"] = df_lista_pd["agencia"].str.strip()

In [106]:
df_resumen = (
    df_lista_pd
    .groupby("estado")
    .size()
    .reset_index(name="cantidad")
)

print(df_resumen)

    estado  cantidad
0   activo       249
1  cargado       128


In [7]:
reemplazos = {
    "PC TACNA": "TACNA",
    "AREQ PAMPILLA": "AREQUIPA PAMPILLA",      # este realmente no cambia
    "AREQ CAYMA": "AREQUIPA CAYMA",
    "ENMANCIPACION": "EMANCIPACION",
    "SAN JUAN DE LURIG": "SAN JUAN DE LURIGANCHO",
    "TRUJ CENTRO": "TRUJILLO CENTRO",
    "TRUJ AMERICA": "TRUJILLO AMERICA",
    "PC HUANCAYO": "HUANCAYO",
    "PC HUARAZ": "HUARAZ",
    "PAITA": "SULLANA",
}

df_lista_pd["agencia"] = df_lista_pd["agencia"].replace(reemplazos)

In [8]:
query = f"""
select agencia_Formulario,agencia_correo  as agencia from Alice.agencias_alfin
"""
df_Age = pd.read_sql(query, engine_mysql)

In [9]:
df_lista_pd_01=df_lista_pd.merge(df_Age,on='agencia',how='left')

In [10]:
df_lista_pd_01.loc[
    df_lista_pd_01["agencia_Formulario"].isna(),
    "agencia"
].unique()

array([], dtype=object)

In [11]:
query = f"""
	SELECT dni_cliente as dni,estado  as estdo_formulario, DATE(fecha_envio) as fecha_envio_formulario FROM Alice.prospectos_envio_alfin 
    where DATE(fecha_envio)>='2026-08-01'
"""
df_formulario_alfin = pd.read_sql(query, engine_mysql)

query = f"""
	SELECT dni_cliente as dni,estado as estdo_correo,DATE(fecha_envio) as fecha_envio_correo FROM Alice.prospectos_correos_alfin 
    where DATE(fecha_envio)>='2026-08-01'
"""
df_correos_alfin = pd.read_sql(query, engine_mysql)


In [12]:
df_formulario_alfin["fecha_envio_formulario"] = pd.to_datetime(
    df_formulario_alfin["fecha_envio_formulario"]
)

df_formulario_alfin = (
    df_formulario_alfin
    .sort_values(
        "fecha_envio_formulario",
        ascending=False
    )
    .drop_duplicates(
        subset="dni",
        keep="first"
    )
)
df_correos_alfin["fecha_envio_correo"] = pd.to_datetime(
    df_correos_alfin["fecha_envio_correo"]
)

df_correos_alfin = (
    df_correos_alfin
    .sort_values(
        "fecha_envio_correo",
        ascending=False
    )
    .drop_duplicates(
        subset="dni",
        keep="first"
    )
)

In [13]:
df_lista_pd_02=df_lista_pd_01.merge(df_formulario_alfin,on='dni',how='left')
df_lista_pd_02=df_lista_pd_02.merge(df_correos_alfin,on='dni',how='left')



In [14]:
df_lista_pd_02 = df_lista_pd_02.drop_duplicates(
    subset="dni",
    keep="last"
)

In [15]:
import numpy as np
import pandas as pd

# Convertir a numérico
df_lista_pd_02["OFERTA_MAX"] = pd.to_numeric(df_lista_pd_02["OFERTA_MAX"], errors="coerce")

bins = [0, 5000, 10000, 15000, 20000, 25000, 30000, np.inf]

labels = [
    "01. [0 - 5,000)",
    "02. [5,000 - 10,000)",
    "03. [10,000 - 15,000)",
    "04. [15,000 - 20,000)",
    "05. [20,000 - 25,000)",
    "06. [25,000 - 30,000)",
    "07. >= 30,000"
]

df_lista_pd_02["RANGO_OFERTA"] = pd.cut(
    df_lista_pd_02["OFERTA_MAX"],
    bins=bins,
    labels=labels,
    right=False
)

In [16]:
filename='TARGET.txt'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_target_desembolso = pd.read_csv(ruta_archivo,sep='|')

# df_target_desembolso.rename(columns={'DNI': 'dni_cliente'}, inplace=True)
# df_target_desembolso.rename(columns={'FECHA_DESEMBOLSOS': 'fecha_desembolso'}, inplace=True)
df_target_desembolso = df_target_desembolso[['DNI']].copy()
df_target_desembolso['target'] = 1

filename='ACUM_DESEM.txt'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_fugas = pd.read_csv(ruta_archivo,sep='|')
df_fugas = df_fugas[['DNI','MONTO','ASESOR','CANALVENTA']].copy()

df_desembolso = df_target_desembolso[['DNI']].drop_duplicates().merge(
    df_fugas[['DNI']].drop_duplicates(),
    on=['DNI'],
    how='inner'
)

df_desembolso = df_target_desembolso.drop_duplicates().merge(
    df_fugas.drop_duplicates(),
    on=['DNI'],
    how='inner'
)
df_desembolso['target'] = df_desembolso['target'].fillna(0).astype(int)
df_desembolso['fugas'] = (df_desembolso['target'] == 0).astype(int)
df_desembolso.rename(columns={'DNI': 'dni_cliente'}, inplace=True)

df_desembolso['dni_cliente'] = (
    df_desembolso['dni_cliente']
    .astype(str)
    .str.replace(r'\D', '', regex=True)   
    .replace('', pd.NA)                     
    .str.zfill(8)                           
)



In [17]:
filename='RetiroDefinitivo_BlackList.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_def_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_BlackList.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_Telefonos.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_telf = pd.read_csv(ruta_archivo,sep='|')
filename='retiro_correo_alfin.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_retiro_correo = pd.read_csv(ruta_archivo,sep=';')

# filename='desembolso.csv'
# ruta_archivo = os.path.join(ruta_alfin, filename)
# df_des = pd.read_csv(ruta_archivo,sep=';')

df_def_blacklist = df_def_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_blacklist = df_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_telf= df_telf.rename(columns={'TELEFONO': 'celular'})
df_retiro_correo= df_retiro_correo.rename(columns={'DNI': 'dni_cliente'})


# Blacklists de DNI
# df6 = df_des.copy()
# df6["celular"] = None
# df6 = df6[["dni_cliente", "celular"]]

# Blacklists de DNI
df1 = df_def_blacklist.copy()
df1["celular"] = None
df1 = df1[["dni_cliente", "celular"]]

df2 = df_blacklist.copy()
df2["celular"] = None
df2 = df2[["dni_cliente", "celular"]]

# Blacklist de teléfonos
df3 = df_telf.copy()
df3["dni_cliente"] = None
df3 = df3[["dni_cliente", "celular"]]

# Archivo con DNI y celular
df4 = df_retiro_correo[["dni_cliente", "celular"]].copy()




In [18]:

# Unir todo
df_retiros = pd.concat(
    [df1, df2, df3, df4],
    ignore_index=True
)

dni_retiro = set(df_retiros['dni_cliente'].dropna())
cel_retiro = set(df_retiros['celular'].dropna())
dni_desembolso = set(df_desembolso['dni_cliente'].dropna())


C:\Users\DATA\AppData\Local\Temp\ipykernel_18468\732878998.py:2: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_retiros = pd.concat(


In [19]:
df_lista_pd_02 = df_lista_pd_02[
    (~df_lista_pd_02["dni"].isin(dni_retiro)) &
    (~df_lista_pd_02["dni"].isin(dni_desembolso)) &
    (~df_lista_pd_02["celular"].isin(cel_retiro))
].copy()

In [21]:
print(df_lista_pd_02.columns.tolist())

['dni', 'nombre', 'celular', 'agencia', 'hoja', 'COLOR_FINAL', 'COD_USER_V3', 'USER_V3', 'PERFIL_RO', 'campaña', 'OFERTA_MAX', 'PLAZO', 'CAPACIDAD_MAX', 'FRESCURA', 'rango_deuda', 'numentidades', 'TOTAL_A_LIQUIDAR', 'TASA_CREDITO_ANTERIOR', 'TASA_1', 'TASA_2', 'TASA_3', 'TASA_4', 'TASA_5', 'TASA_6', 'TASA_7', 'MGNEG', 'MARCA_PD', 'AUTORIZACION_DATOS', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'TIPO_BASE', 'PROPENSION_DISTRIBUCION', 'OFERTA_SS', 'TASA_1_SS', 'TASA_2_SS', 'TASA_3_SS', 'TASA_4_SS', 'TASA_5_SS', 'TASA_6_SS', 'TASA_7_SS', 'ALERTA_MAQUETA', 'FEN', 'PERFIL_ESPECIAL', 'TIPO', 'tipo_archivo', 'agencia_Formulario', 'estdo_formulario', 'fecha_envio_formulario', 'estdo_correo', 'fecha_envio_correo', 'RANGO_OFERTA']


In [20]:

ruta_archivo = os.path.join(ruta_csv, 'resumen_envio.csv')
df_lista_pd_02.to_csv(ruta_archivo,sep=';')

In [61]:
df_lista_pd_02.head(2)

,nombre,celular,agencia,hoja,COLOR_FINAL,COD_USER_V3,USER_V3,PERFIL_RO,campaña,OFERTA_MAX,...,FEN,PERFIL_ESPECIAL,TIPO,tipo_archivo,agencia_Formulario,estdo_formulario,fecha_envio_formulario,estdo_correo,fecha_envio_correo,RANGO_OFERTA
0,MELCHOR ANTONIO MERCADO RUIZ,950582801,PUCALLPA,1,NARANJA OSCURO,3,3. MES + PLD Peers,Aceptante I,SOLODNI,8000.0,...,None,None,PRESTALTOKE,campo,738334 - PUCALLPA,NaN,NaT,NaN,NaT,"02. [5,000 - 10,000)"
1,OSWALDO GONZALES CURINUQUI,939173565,TARAPOTO,1,VERDE OSCURO,3,3. MES + PLD Peers,Inconforme II,SOLODNI,14200.0,...,None,None,PRESTALTOKE,campo,733824 - TARAPOTO,NaN,NaT,NaN,NaT,"03. [10,000 - 15,000)"


In [ ]:
# df=df_subir.toPandas()
query = f"""
select agencia_Formulario,agencia_correo from Alice.agencias_alfin
"""
df_Age = pd.read_sql(query, engine_mysql)
ruta_archivo = os.path.join(ruta_csv, 'SOOOOO.csv')
df_Age.to_csv(ruta_archivo, index=False)

'PC TACNA', 'AREQ PAMPILLA', 'ENMANCIPACION', 'SAN JUAN DE LURIG',
       'AREQ CAYMA', 'TRUJ CENTRO', 'TRUJ AMERICA', 'PC HUANCAYO',
       'PC HUARAZ', 'PAITA'

In [20]:
df_lista_pd_01[df_lista_pd_01['agencia_Formulario'].isna()].head()


,dni,nombre,celular,agencia,hoja,COLOR_FINAL,COD_USER_V3,USER_V3,PERFIL_RO,campaña,...,TASA_4_SS,TASA_5_SS,TASA_6_SS,TASA_7_SS,ALERTA_MAQUETA,FEN,PERFIL_ESPECIAL,TIPO,tipo_archivo,agencia_Formulario
45,00401050,JUAN CARLOS SUAREZ COHAILA,984495438,PC TACNA,1,VERDE CLARO,7,7. Peers,Aceptante II,SOLO CON DNI - ZONA COBERTURA,...,65,62,56,56,None,None,None,PRESTALTOKE,campo,NaN
46,00405457,MARIA ESTER AROS DE CARNERO,952079384,PC TACNA,1,NARANJA CLARO,6,6. MES B,Aceptante I,SOLODNI,...,110,110,110,110,None,None,None,PRESTALTOKE,campo,NaN
47,00418793,BLANCA ISABEL VASCONES DE PALZA,976563114,PC TACNA,1,VERDE OSCURO,8,8. Tarjetero Cash,Inconforme II,MUJER,...,91,91,91,91,None,None,None,PRESTALTOKE,campo,NaN
48,00426087,PATRICIA FERNANDA YESQUEN OTTONE,949200194,PC TACNA,1,NARANJA CLARO,14,14. Otros Bancarizados,Inconforme I,SOLODNI,...,110,110,110,110,None,None,None,PRESTALTOKE,campo,NaN
49,00429633,VALDIVIA GAMBOA WILBER RAFAEL,965659905,PC TACNA,1,AMARILLO CLARO,2,2. sunedu & sunarp B,Resistente II,SOLODNI,...,83,81,73,73,None,None,None,PRESTALTOKE,campo,NaN


In [3]:
filename='usar_01.csv'
df_subir=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)

In [5]:
df_subir=df_subir.withColumnRenamed('dni_cliente','DNI')

In [4]:
df_base=df_base.drop( 'COLOR_FINAL', 'COD_USER_V3', 'USER_V3', 'PERFIL_RO', 'campaña', 'OFERTA_MAX', 'PLAZO', 'CAPACIDAD_MAX', 'FRESCURA', 'rango_deuda', 'numentidades', 'TOTAL_A_LIQUIDAR', 'TASA_CREDITO_ANTERIOR', 'TASA_1', 'TASA_2', 'TASA_3', 'TASA_4', 'TASA_5', 'TASA_6', 'TASA_7', 'MGNEG', 'MARCA_PD', 'AUTORIZACION_DATOS', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'TIPO_BASE', 'PROPENSION_DISTRIBUCION', 'OFERTA_SS', 'TASA_1_SS', 'TASA_2_SS', 'TASA_3_SS', 'TASA_4_SS', 'TASA_5_SS', 'TASA_6_SS', 'TASA_7_SS', 'ALERTA_MAQUETA', 'FEN', 'PERFIL_ESPECIAL', 'TIPO', 'tipo_archivo')

In [5]:
filename='descartar!11.csv'
df_quitar_pues=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)


In [8]:
df_validar = df_validar.withColumn(
    "DNI",
    F.right(
        F.concat(F.lit("00000000"), F.col("DNI")),
        F.lit(8)
    )

)

In [9]:
df_subir=df_subir.join(df_validar,['DNI'],'inner')

In [10]:
df_subir=df_subir.dropDuplicates(['DNI'])

In [94]:


filename='desembolso_alfin.csv'
desembolso_quitar=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)
desembolso_quitar=desembolso_quitar.withColumnRenamed('Número Documento','DNI')
desembolso_quitar.show(2)

+------+----------------+--------------+--------------+----------+-----------+----------------+------+----------------+-----------------+------------+--------------------+--------------------+------+--------+--------------+---------+------------+-----+------------------+-------------+-------------+-------------------+---------------+-----------+-----+-------------+----------+---+-----+-----+------+-----------+-------------+-----+------+--------------------+--------------------+------------+--------------------+----------+--------------+--------------+----------------------+------------+-----------------+---------------------+-------------+--------------------+-------------------+--------+---------+------------------+
|AñoMes|Fecha Desembolso|Canal Venta BT|Canal de Venta|    Región|CodSuc (BT)|   Sucursal (BT)|CodSuc|        Sucursal|Cod Empleado (BT)|Cod Empleado|          Nombre ANC|         Vendor (BT)|Vendor|     DNI|Cuenta Cliente|Operación|     Nombres|  Mda|Capital Solicitado|To

In [95]:
df_quitar_pues=df_quitar_pues.withColumnRenamed('dni_cliente','DNI')

In [ ]:
df_base_01

In [37]:
desembolso_quitar.count()

9254

In [ ]:


filename='BASE_CEL_TARGET_20260801.txt'
df_score=cargar_archivo_csv_ruta(spark,filename,'|',True,ruta_alfin)

filename='fomato_agendas_alfin_credicash_2026.csv'
formato_agendas=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)

filename='RetiroDefinitivo_BlackList.csv'
df_def_blacklist=cargar_archivo_csv_ruta(spark,filename,'|',True,ruta_alfin)
filename='RetiroDeGestion_BlackList.csv'
df_blacklist=cargar_archivo_csv_ruta(spark,filename,'|',True,ruta_alfin)
filename='RetiroDeGestion_Telefonos.csv'
df_retirogestion=cargar_archivo_csv_ruta(spark,filename,'|',True,ruta_alfin)
filename='retiro_correo_alfin.csv'
df_retiro_correo=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)

filename='agencia_alfin_cod.csv'
df_agencia_alfin=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)

print(df_base.columns)
print(df_score.columns)
print(df_validar.columns)
print(formato_agendas.columns)
print(df_def_blacklist.columns)
print(df_blacklist.columns)
print(df_retirogestion.columns)
print(df_retiro_correo.columns)
print(df_agencia_alfin.columns)

['PROPENSION_IC', 'DNI', 'X_APPATERNO', 'X_APMATERNO', 'X_NOMBRE', 'tasa_minima', 'CUOTA', 'TIPO_GEST', 'TIPO_CLIENTE_COMERCIAL', 'SALDO_DIFERENCIAL_REENG', 'TIPO_CLIENTE', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'SUCURSAL_COMERCIAL', 'Agencia_comercial', 'REGION_COMERCIAL', 'VARIACION_OFERTA_CAMPAÑA_ANTERIOR', 'VARIACION_TASA_CAMPAÑA_ANTERIOR', 'VAR_TASA_CREDITO_ANTERIOR', 'FLG_CET_6M', 'GRUPO_MONTO', 'Edad', 'RANGO_EDAD', 'RANGO_OFERTA', 'RANGO_SUELDO', 'BLOQUE', 'FLG_AAHH', 'INTENSIDAD_MAX', 'PILOTO_PLAZAS', 'CAMP_BONO', 'CAMP_ADP', 'EXCLUSIVO', 'PILOTO_RETENCION', 'ACCION']
['DNI', 'CET', 'CEL1', 'SCORE_TELEFONO']
['DNI', 'COLOR_FINAL', 'COD_USER_V3', 'USER_V3', 'PERFIL_RO', 'campaña', 'OFERTA_MAX', 'PLAZO', 'CAPACIDAD_MAX', 'FRESCURA', 'rango_deuda', 'numentidades', 'TOTAL_A_LIQUIDAR', 'TASA_CREDITO_ANTERIOR', 'TASA_1', 'TASA_2', 'TASA_3', 'TASA_4', 'TASA_5', 'TASA_6', 'TASA_7', 'MGNEG', 'MARCA_PD', 'AUTORIZACION_DATOS', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'TIPO_BASE', 'PROPENSI

In [ ]:
['PROPENSION_IC', 'USER_V3', 'DNI', 'X_APPATERNO', 'X_APMATERNO', 'X_NOMBRE', 'OFERTA_MAX', 'tasa_minima', 'PLAZO', 'CUOTA', 'CAPACIDAD_MAX', 'TIPO_GEST', 'TIPO_CLIENTE_COMERCIAL', 'Campaña', 'SALDO_DIFERENCIAL_REENG', 'TIPO_CLIENTE', 'color_final', 'PERFIL_RO', 'TIPO_BASE', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'SUCURSAL_COMERCIAL', 'Agencia_comercial', 'REGION_COMERCIAL', 'VARIACION_OFERTA_CAMPAÑA_ANTERIOR', 'VARIACION_TASA_CAMPAÑA_ANTERIOR', 'VAR_TASA_CREDITO_ANTERIOR', 'FLG_CET_6M', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'GRUPO_MONTO', 'MGNEG', 'Tasa_1', 'Tasa_2', 'Tasa_3', 'Tasa_4', 'Tasa_5', 'Tasa_6', 'Tasa_7', 'Edad', 'RANGO_EDAD', 'RANGO_OFERTA', 'RANGO_SUELDO', 'BLOQUE', 'FRESCURA', 'FLG_AAHH', 'INTENSIDAD_MAX', 'PILOTO_PLAZAS', 'CAMP_BONO', 'CAMP_ADP', 'EXCLUSIVO', 'PILOTO_RETENCION', 'ACCION']acc


In [59]:
df_base.dropDuplicates(['DNI']).count()

175755

In [7]:
df_quitar_pues.show(3)

+---+-----------+
|_c0|dni_cliente|
+---+-----------+
|  0|   48124718|
|  1|   41494656|
|  2|   41298318|
+---+-----------+
only showing top 3 rows


In [8]:
df_quitar_pues=df_quitar_pues.withColumnRenamed('dni_cliente','DNI')


In [9]:
df_base_01=df_base.join(df_validar,['DNI'],'inner')
df_base_01=df_base.join(df_quitar_pues,['DNI'],'left_anti')
df_base_01=df_base_01.dropDuplicates(['DNI'])

In [61]:
formato_agendas.show(3)

+-----------+--------------------+---------+--------------------+----------------+------------+-----+-------+
|dni_cliente|      nombre_cliente|  celular|         cod_agencia|agencia_atencion|fecha_visita|monto|color_1|
+-----------+--------------------+---------+--------------------+----------------+------------+-----+-------+
|    6757927|QUIROZ MONCADA SA...|987613889|738382 - JESUS MARIA|            NULL|        NULL|20000|   NULL|
|     426135|María asunta cáce...|969562424|   738013 - PC TACNA|            NULL|        NULL| 9900|   NULL|
|   22488348|Patricia Mabel Me...|962578360|    735996 - HUANUCO|            NULL|        NULL|16400|   NULL|
+-----------+--------------------+---------+--------------------+----------------+------------+-----+-------+
only showing top 3 rows


In [62]:
formato_agendas=formato_agendas.withColumnRenamed('dni_cliente','DNI')

In [63]:
formato_agendas1=formato_agendas.join(df_validar,['DNI'],'inner')
formato_agendas1=formato_agendas1.dropDuplicates(['DNI'])
formato_agendas1.count()

13035

In [ ]:
formato_agendas1=formato_agendas1

In [51]:
df=formato_agendas1.toPandas()

In [64]:
ruta_archivo = os.path.join(ruta_csv, 'consulta_campana_2.xlsx')
df.to_excel(ruta_archivo, index=False)

In [16]:
# df=df_subir.toPandas()
query = f"""
select agencia_Formulario,agencia_correo from Alice.agencias_alfin
"""
df_Age = pd.read_sql(query, engine_mysql)
ruta_archivo = os.path.join(ruta_csv, 'SOOOOO.csv')
df_Age.to_csv(ruta_archivo, index=False)

In [79]:
print(df_base_01.columns)

['DNI', 'Agencia_comercial', 'COLOR_FINAL', 'COD_USER_V3', 'USER_V3', 'PERFIL_RO', 'campaña', 'OFERTA_MAX', 'PLAZO', 'CAPACIDAD_MAX', 'FRESCURA', 'rango_deuda', 'numentidades', 'TOTAL_A_LIQUIDAR', 'TASA_CREDITO_ANTERIOR', 'TASA_1', 'TASA_2', 'TASA_3', 'TASA_4', 'TASA_5', 'TASA_6', 'TASA_7', 'MGNEG', 'MARCA_PD', 'AUTORIZACION_DATOS', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'TIPO_BASE', 'PROPENSION_DISTRIBUCION', 'OFERTA_SS', 'TASA_1_SS', 'TASA_2_SS', 'TASA_3_SS', 'TASA_4_SS', 'TASA_5_SS', 'TASA_6_SS', 'TASA_7_SS', 'ALERTA_MAQUETA', 'FEN', 'PERFIL_ESPECIAL', 'TIPO', 'tipo_archivo', 'CET', 'CEL1', 'SCORE_TELEFONO']


In [10]:
df_base_01=df_base_01.join(df_score,['DNI'],'inner')

In [ ]:
['DNI', 'Agencia_comercial', 'COLOR_FINAL', 'COD_USER_V3', 'USER_V3', 'PERFIL_RO', 'campaña', 'OFERTA_MAX', 'PLAZO', 'CAPACIDAD_MAX', 'FRESCURA', 'rango_deuda', 'numentidades', 'TOTAL_A_LIQUIDAR', 'TASA_CREDITO_ANTERIOR', 'TASA_1', 'TASA_2', 'TASA_3', 'TASA_4', 'TASA_5', 'TASA_6', 'TASA_7', 'MGNEG', 'MARCA_PD', 'AUTORIZACION_DATOS', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'TIPO_BASE', 'PROPENSION_DISTRIBUCION', 'OFERTA_SS', 'TASA_1_SS', 'TASA_2_SS', 'TASA_3_SS', 'TASA_4_SS', 'TASA_5_SS', 'TASA_6_SS', 'TASA_7_SS', 'ALERTA_MAQUETA', 'FEN', 'PERFIL_ESPECIAL', 'TIPO', 'tipo_archivo', 'CET', 'CEL1', 'SCORE_TELEFONO']lo

In [ ]:
# grupo_tasa=['None', '', '', '', '', ]
# agencia_comercial=[ 'SAN MIGUEL', 'MIRAFLORES']

df_filtrado = df_base_01.filter(
    # (F.col('PROPENSION_DISTRIBUCION').isin('1','2','3'))&
    # (F.col('proveedor').isin(['INVENTARIO TARGET 2', 'CET TARGET 2', 'CET TARGET 0', 'INVENTARIO TARGET 1', 'INVENTARIO TARGET 0', ]))&
    # (F.col('user_v3').isin(user_v3))&

    # (

    (
        (F.col('USER_V3').isin('1. sunedu & sunarp A','2. sunedu & sunarp B','3. MES + PLD Peers'))&
        (F.col('TIPO_CLIENTE')=='INDEPENDIENTE')
    )
        
        (F.col('OFERTA_MAX')>=10000)&
        (F.col('lote')!='BOT')
    (F.col('retiro')=='ACTIVO'))

)



print(df_filtrado.count())
print(df_filtrado.columns)

In [44]:
formato_agendas.count()

22718

In [47]:
formato_agendas.dropDuplicates(['dni_cliente']).count()

22718

In [ ]:
df_base_telf=df_base_telf

In [39]:
df_base_01=df_base.filter(F.col('Agencia_comercial').isNotNull())
df_base_01=df_base_01.select('DNI','Agencia_comercial')

In [80]:
print([row['ACCION' ] for row in df_base.select('ACCION').distinct().collect()])


['NO CLIENTE INTERESADO OPERADOR', 'NO CLIENTE TICKET <= 5K ALTA PROPENSION', 'NO CLIENTE CONTACTADO OPERADOR', 'NO CLIENTE RESTO']


In [82]:
df_base=df_base.join(df_quitar_pues,['DNI'],'left_anti')

In [84]:
df_base.groupBy('ACCION') \
    .count() \
    .orderBy('ACCION') \
    .show(30,truncate=False)


+---------------------------------------+------+
|ACCION                                 |count |
+---------------------------------------+------+
|NO CLIENTE CONTACTADO OPERADOR         |10333 |
|NO CLIENTE INTERESADO OPERADOR         |3615  |
|NO CLIENTE RESTO                       |55485 |
|NO CLIENTE TICKET <= 5K ALTA PROPENSION|105902|
+---------------------------------------+------+



In [ ]:
df_base_02=df_base.join(df_validar,['DNI'],'inner')


In [11]:
# df_base_01=df_base.join(df_validar,['DNI'],'inner')
df_base_01=df_base_01.filter(F.col('ACCION')=='NO CLIENTE INTERESADO OPERADOR')
df_base_01=df_base_01.dropDuplicates(['DNI'])
df_base_01.count()

3615

In [12]:
df_base_03=df_base_01.filter(F.col('OFERTA_MAX')>=5000)
df_base_03.count()

{"ts": "2026-08-01 10:13:35.525", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `OFERTA_MAX` cannot be resolved. Did you mean one of the following? [`CET`, `CUOTA`, `FLG_AAHH`, `CEL1`, `DISTRITO`]. SQLSTATE: 42703", "context": {"file": "java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o180.filter.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `OFERTA_MAX` cannot be resolved. Did you mean one of the following? [`CET`, `CUOTA`, `FLG_AAHH`, `CEL1`, `DISTRITO`]. SQLSTATE: 42703;\n'Filter '`>=`('OFERTA_MAX, 5000)\n+- Deduplicate [DNI#19]\n   +- Filter (ACCION#70 = NO CLIENTE INTERESADO OPERADOR)\n      +- Pro

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `OFERTA_MAX` cannot be resolved. Did you mean one of the following? [`CET`, `CUOTA`, `FLG_AAHH`, `CEL1`, `DISTRITO`]. SQLSTATE: 42703;
'Filter '`>=`('OFERTA_MAX, 5000)
+- Deduplicate [DNI#19]
   +- Filter (ACCION#70 = NO CLIENTE INTERESADO OPERADOR)
      +- Project [DNI#19, PROPENSION_IC#17, X_APPATERNO#20, X_APMATERNO#21, X_NOMBRE#22, tasa_minima#24, CUOTA#26, TIPO_GEST#28, TIPO_CLIENTE_COMERCIAL#29, SALDO_DIFERENCIAL_REENG#31, TIPO_CLIENTE#32, DEPARTAMENTO#36, PROVINCIA#37, DISTRITO#38, SUCURSAL_COMERCIAL#39, Agencia_comercial#40, REGION_COMERCIAL#41, VARIACION_OFERTA_CAMPAÑA_ANTERIOR#42, VARIACION_TASA_CAMPAÑA_ANTERIOR#43, VAR_TASA_CREDITO_ANTERIOR#44, FLG_CET_6M#45, GRUPO_MONTO#48, Edad#57, RANGO_EDAD#58, RANGO_OFERTA#59, ... 13 more fields]
         +- Join Inner, (DNI#19 = DNI#346)
            :- Deduplicate [DNI#19]
            :  +- Project [DNI#19, PROPENSION_IC#17, X_AP

In [ ]:
print(df_base_02.columns)['DNI', 'PROPENSION_IC', 'USER_V3', 'X_APPATERNO', 'X_APMATERNO', 'X_NOMBRE', 'OFERTA_MAX', 'tasa_minima', 'PLAZO', 'CUOTA', 'CAPACIDAD_MAX', 'TIPO_GEST', 'TIPO_CLIENTE_COMERCIAL', 'Campaña', 'SALDO_DIFERENCIAL_REENG', 'TIPO_CLIENTE', 'color_final', 'PERFIL_RO', 'TIPO_BASE', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'SUCURSAL_COMERCIAL', 'Agencia_comercial', 'REGION_COMERCIAL', 'VARIACION_OFERTA_CAMPAÑA_ANTERIOR', 'VARIACION_TASA_CAMPAÑA_ANTERIOR', 'VAR_TASA_CREDITO_ANTERIOR', 'FLG_CET_6M', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'GRUPO_MONTO', 'MGNEG', 'Tasa_1', 'Tasa_2', 'Tasa_3', 'Tasa_4', 'Tasa_5', 'Tasa_6', 'Tasa_7', 'Edad', 'RANGO_EDAD', 'RANGO_OFERTA', 'RANGO_SUELDO', 'BLOQUE', 'FRESCURA', 'FLG_AAHH', 'INTENSIDAD_MAX', 'PILOTO_PLAZAS', 'CAMP_BONO', 'CAMP_ADP', 'EXCLUSIVO', 'PILOTO_RETENCION', 'ACCION', 'COLOR_FINAL', 'COD_USER_V3', 'USER_V3', 'PERFIL_RO', 'campaña', 'OFERTA_MAX', 'PLAZO', 'CAPACIDAD_MAX', 'FRESCURA', 'rango_deuda', 'numentidades', 'TOTAL_A_LIQUIDAR', 'TASA_CREDITO_ANTERIOR', 'TASA_1', 'TASA_2', 'TASA_3', 'TASA_4', 'TASA_5', 'TASA_6', 'TASA_7', 'MGNEG', 'MARCA_PD', 'AUTORIZACION_DATOS', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'TIPO_BASE', 'PROPENSION_DISTRIBUCION', 'OFERTA_SS', 'TASA_1_SS', 'TASA_2_SS', 'TASA_3_SS', 'TASA_4_SS', 'TASA_5_SS', 'TASA_6_SS', 'TASA_7_SS', 'ALERTA_MAQUETA', 'FEN', 'PERFIL_ESPECIAL', 'TIPO', 'tipo_archivo']ofe

['DNI', 'PROPENSION_IC', 'USER_V3', 'X_APPATERNO', 'X_APMATERNO', 'X_NOMBRE', 'OFERTA_MAX', 'tasa_minima', 'PLAZO', 'CUOTA', 'CAPACIDAD_MAX', 'TIPO_GEST', 'TIPO_CLIENTE_COMERCIAL', 'Campaña', 'SALDO_DIFERENCIAL_REENG', 'TIPO_CLIENTE', 'color_final', 'PERFIL_RO', 'TIPO_BASE', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'SUCURSAL_COMERCIAL', 'Agencia_comercial', 'REGION_COMERCIAL', 'VARIACION_OFERTA_CAMPAÑA_ANTERIOR', 'VARIACION_TASA_CAMPAÑA_ANTERIOR', 'VAR_TASA_CREDITO_ANTERIOR', 'FLG_CET_6M', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'GRUPO_MONTO', 'MGNEG', 'Tasa_1', 'Tasa_2', 'Tasa_3', 'Tasa_4', 'Tasa_5', 'Tasa_6', 'Tasa_7', 'Edad', 'RANGO_EDAD', 'RANGO_OFERTA', 'RANGO_SUELDO', 'BLOQUE', 'FRESCURA', 'FLG_AAHH', 'INTENSIDAD_MAX', 'PILOTO_PLAZAS', 'CAMP_BONO', 'CAMP_ADP', 'EXCLUSIVO', 'PILOTO_RETENCION', 'ACCION', 'COLOR_FINAL', 'COD_USER_V3', 'USER_V3', 'PERFIL_RO', 'campaña', 'OFERTA_MAX', 'PLAZO', 'CAPACIDAD_MAX', 'FRESCURA', 'rango_deuda', 'numentidades', 'TOTAL_A_LIQUIDAR', 'TASA_CREDITO_A

In [ ]:

df_filtrado=df_filtrado.withColumnRenamed('NUMERO_DOCUMENTO','vendor_lead_code')
# df_filtrado=df_filtrado.withColumnRenamed('cl_telf1','phone_number')
# df_filtrado=df_filtrado.withColumnRenamed('NOMBRES','address1')

df_filtrado = df_filtrado.withColumn(
    "address1",
    F.concat_ws(
        " ",
        F.col("NOMBRES"),
        F.col("APELLIDO_PATERNO"),
        F.col("APELLIDO_MATERNO")
    )
)

df_filtrado = df_filtrado.withColumn(
    "security_phrase",
    F.concat_ws(
        " ",
        F.lit("Oferta:"),
        F.col("OFERTA_MAX")
    )
)
df_filtrado = df_filtrado.withColumn(
    "city",
    F.concat_ws(
        " ",
        F.lit("Departamento:"),
        F.col("DEPARTAMENTO")
    )
)
df_filtrado = df_filtrado.withColumn(
    "province",
    F.concat_ws(
        " ",
        F.lit("Distrito:"),
        F.col("DISTRITO")
    )
)

df_filtrado.count()

In [81]:
df_base.groupBy('ACCION') \
    .count() \
    .orderBy('ACCION') \
    .show(30)


+--------------------+------+
|              ACCION| count|
+--------------------+------+
|NO CLIENTE CONTAC...| 10394|
|NO CLIENTE INTERE...|  3956|
|    NO CLIENTE RESTO| 55503|
|NO CLIENTE TICKET...|105902|
+--------------------+------+



In [ ]:
['LOS OLIVOS', 'VILLA MARIA 2', 'MOSHOQUEQUE', 'PUENTE PIEDRA', 'COMAS', 'PUCALLPA', 'PISCO', 'TARAPOTO', 'CHINCHA', 'VENTANILLA', 'TUMBES', 'VILLA EL SALVADOR 2', 'PC TACNA', 'CAÑETE', 'HUANUCO', 'SANTA ANITA', 'JESUS MARIA', 'EMANCIPACION', 'CAJAMARCA', 'SAN MARTIN', 'SAN MIGUEL', 'MIRAFLORES', 'AREQUIPA CAYMA', 'SULLANA', 'AREQUIPA PAMPILLA', 'IQUITOS', 'ATE VITARTE', 'CHICLAYO BALTA', 'HUACHO', 'SAN JUAN DE MIRAFLORES', 'SAN JUAN DE LURIGANCHO', 'PC HUARAZ', 'PC HUANCAYO', 'CASTILLA', 'ICA', 'TRUJILLO CENTRO', 'CHIMBOTE', 'HUARAL', 'JULIACA 2', 'TRUJILLO AMERICA', 'CUSCO LA CULTURA', None]cañ

+--------+---------------+-----------+--------------------+-------------+-------+----------+-----+-------------+--------+-----------+------------+----------------+---------------------+------+------+------+------+------+------+------+-----+--------+------------------+-------------------+-------------+----------------+-----------------------+---------+---------+---------+---------+---------+---------+---------+---------+--------------+----+---------------+-----------+------------+
|     DNI|    COLOR_FINAL|COD_USER_V3|             USER_V3|    PERFIL_RO|campaña|OFERTA_MAX|PLAZO|CAPACIDAD_MAX|FRESCURA|rango_deuda|numentidades|TOTAL_A_LIQUIDAR|TASA_CREDITO_ANTERIOR|TASA_1|TASA_2|TASA_3|TASA_4|TASA_5|TASA_6|TASA_7|MGNEG|MARCA_PD|AUTORIZACION_DATOS|FLAG_DEUDA_V_OFERTA|   GRUPO_TASA|       TIPO_BASE|PROPENSION_DISTRIBUCION|OFERTA_SS|TASA_1_SS|TASA_2_SS|TASA_3_SS|TASA_4_SS|TASA_5_SS|TASA_6_SS|TASA_7_SS|ALERTA_MAQUETA| FEN|PERFIL_ESPECIAL|       TIPO|tipo_archivo|
+--------+---------------+----

In [48]:
formato_agendas=formato_agendas.withColumnRenamed('dni_cliente','DNI')

In [98]:

def completar_dni(df):
    return df.withColumn(
        "DNI",
        F.lpad(F.col("DNI").cast("string"), 8, "0")
    )

df_base_01 = completar_dni(df_base_01)
df_validar = completar_dni(df_validar)
df_def_blacklist = completar_dni(df_def_blacklist)
df_blacklist = completar_dni(df_blacklist)
# df_retiro_correo = completar_dni(df_retiro_correo)
formato_agendas = completar_dni(formato_agendas)
df_quitar_pues = completar_dni(df_quitar_pues)
df_score = completar_dni(df_score)

{"ts": "2026-08-01 10:08:01.603", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `DNI` cannot be resolved. Did you mean one of the following? [`monto`, `celular`, `color_1`, `cod_agencia`, `dni_cliente`]. SQLSTATE: 42703", "context": {"file": "line 4 in cell [99]", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o1883.withColumn.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `DNI` cannot be resolved. Did you mean one of the following? [`monto`, `celular`, `color_1`, `cod_agencia`, `dni_cliente`]. SQLSTATE: 42703;\n'Project [dni_cliente#10623, nombre_cliente#10624, celular#10625, cod_agencia#10626, agencia_atencion#10627, fecha_visita#10628, monto#10629, color_1#10630, 'lpad(cast('DNI

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `DNI` cannot be resolved. Did you mean one of the following? [`monto`, `celular`, `color_1`, `cod_agencia`, `dni_cliente`]. SQLSTATE: 42703;
'Project [dni_cliente#10623, nombre_cliente#10624, celular#10625, cod_agencia#10626, agencia_atencion#10627, fecha_visita#10628, monto#10629, color_1#10630, 'lpad(cast('DNI as string), 8, 0) AS DNI#10739]
+- Relation [dni_cliente#10623,nombre_cliente#10624,celular#10625,cod_agencia#10626,agencia_atencion#10627,fecha_visita#10628,monto#10629,color_1#10630] csv


In [7]:
df_base=df_base.join(df_def_blacklist,['DNI'],'leftanti')
df_base=df_base.join(df_blacklist,['DNI'],'leftanti')
df_base=df_base.join(df_retiro_correo,['DNI'],'leftanti')

In [ ]:

df_base = df_base.withColumnRenamed('X_APPATERNO','APELLIDO_PATERNO')
df_base = df_base.withColumnRenamed('X_APMATERNO','APELLIDO_MATERNO')
df_base = df_base.withColumnRenamed('CampaÃ±a','campania')
df_base = df_base.withColumnRenamed('X_NOMBRE','NOMBRES')
df_base = df_base.withColumnRenamed('DNI','NUMERO_DOCUMENTO')
df_base = df_base.withColumnRenamed('SUCURSAL_COMERCIAL','sucursal_comercial')
df_base = df_base.withColumnRenamed('FLAG_DEUDA_V_OFERTA','flag_deuda_v_oferta')
df_base = df_base.withColumnRenamed('REGION_COMERCIAL','Region_comercial')
df_base = df_base.withColumnRenamed('TIPO_CLIENTE_COMERCIAL','tipo_cliente_riegos')
df_base = df_base.withColumn("Campana", F.lit("202606"))
df_base = df_base.withColumn("lote", F.lit("BASE 2026-06-27"))
df_base = df_base.withColumn("cl_base", F.lit("Junio 2026"))
df_base = df_base.withColumn("cl_carga", F.lit("2026-07-27"))
df_base = df_base.withColumn("cl_estado", F.lit("1"))
df_base = df_base.withColumn("estado", F.lit("ACTIVO"))


In [13]:
query = f"""
    SELECT * FROM DANTALION.[dbo].Base_Maestra_Alfin_bk
    where fecha_envio>='2026-07-01'
    """
df_formato=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)
df_formato.show(2)

+--------+----------------+-----------+----------------+----------------+--------+------------+------------+----------------+--------+--------------+----------+-----------+-----------------+------------+---------+----------+------+------+------+------+------+------+------+------+--------+-------+-----+----+-------------+-----------+-------+----+----------+--------+---------------+---------+----------+--------+---------------+---------+----------+--------+---------------+---------+----------+--------+---------------+---------+------------------+---------+----------------+-------+---------+-------+---------+-------+---------+------------------+-----------------+--------------------+---------+---------------------+-----+------------+----------+------------+--------+-----------------------+------------+------------+-------------+----+----------+---------+-------------+----------+---------+---------+---------+----------+---------+------------+-------------+-------+------------------------+-

In [14]:
from pyspark.sql import functions as F

exprs = [
    F.count(F.when(F.col(c).isNotNull(), c)).alias(c)
    for c in df_base.columns
]

df_counts = df_base.agg(*exprs).collect()[0].asDict()

cols_con_data = [c for c, v in df_counts.items() if v > 0]

df_base = df_base.select(cols_con_data)

In [15]:
en_formato=set(df_formato.columns)
en_base=set(df_base.columns)

In [12]:
print(en_formato-en_base)
print(en_base-en_formato)

{'PROPENSION', 'cl_telf7', 'FLAG_REENG', 'cl_mes', 'Desgravamen_12M', 'Validador_Telefono', 'lote', 'CUOTA_36M', 'NUEVOS_9M', 'NUM_ENRIQUECIDO', 'cl_celular', 'Deuda_1', 'ID_CLIENTE', 'SBI', 'cl_telf5', 'APELLIDO_PATERNO', 'cl_accion', 'campania', 'Desgravamen', 'AÑO_DURACION_BASE', 'Prioridad', 'APELLIDO_MATERNO', 'incremento_monto_riesgos', 'cl_tiempo', 'REP1', 'OFERTA_REEN', 'Entidad_2', 'CUOTA_12M', 'cl_area', 'COD_BD', 'cl_telefono', 'RANGO_EDAD2', 'RETIRO', 'Desgravamen_18M', 'TIENDA', 'TIPO_CONTACTO', 'cl_hits', 'FLAT2', 'REP2', 'CUOTA_24M', 'SEGMENTO_USER', 'cl_turno', 'cl_gestor', 'PROMOCION2', 'sucursal_comercial', 'cl_telf10', 'TIPO_BD', 'CLIENTE_NUEVO', 'RETIRO_GEST', 'OfertaMaximaSinSeguro', 'FECHA_SOL', 'USUARIO', 'cl_carga', 'PERFIL_GLOBAL', 'Desgravamen_24M', 'Oferta_Minima_Paperless', 'Oferta_18M', 'cl_telf1', 'Tasa_18M', 'cl_fecha_llamar', 'Entidad_1', 'cl_telf2', 'cl_hora_gestion', 'NUEVOS_12M', 'marca3', 'NOMBRES', 'color', 'SUCURSAL', 'MONTO_DESEMBOLSADO', 'cl_base

In [ ]:
query = f"""
    SELECT *
        FROM (
            select a.fecha as fecha_llamada,a.dni as NUMERO_DOCUMENTO,a.telefono as cl_telf1,  1 as desembolso,1 as ordern 
            from VALENTINA.dbo.alfcc_ventas a
            inner join cronox.dbo.ref_alfin_dni b
                on a.dni COLLATE Modern_Spanish_CI_AS=b.NUMERO_DOCUMENTO
            where a.estado in(4,13,0)
            and a.fecha<'2026-06-01'
            UNION ALL
            select a.fecha as fecha_llamada,a.dni as NUMERO_DOCUMENTO,a.telefono as cl_telf1,  1 as desembolso,  2 as orden
            from VALENTINA.dbo.alfin_ventas a
            inner join cronox.dbo.ref_alfin_dni b
                on a.dni COLLATE Modern_Spanish_CI_AS=b.NUMERO_DOCUMENTO
            where a.estado in(4,13,0)
            and a.fecha<'2026-06-01'
        ) t
        """
df_desembolso=obtener_tabla_sql(spark,query,server_sa,user_sa,pwd_sa,db_sa)
df_desembolso.count()

In [ ]:
query = f"""
    SELECT *
    FROM (
        SELECT a.NUMERO_DOCUMENTO, a.TELEFONO, a.TIPO_GESTION, a.GESTION, a.FECHA_ENVIO, 6 AS peso
        FROM [MAEBA].[ADM_OBJ_TG].[tGestionMesCencosudTc] a
        INNER JOIN odin.dbo.ref_alfin_dni b
            ON a.NUMERO_DOCUMENTO = b.NUMERO_DOCUMENTO
        WHERE a.TELEFONO IS NOT NULL

        UNION ALL

        SELECT a.NUMERO_DOCUMENTO, a.TELEFONO, a.TIPO_GESTION, a.GESTION, a.FECHA_ENVIO, 4 AS peso
        FROM [MAEBA].[ADM_OBJ_TG].tGestionMesDiners a
        INNER JOIN odin.dbo.ref_alfin_dni b
            ON a.NUMERO_DOCUMENTO = b.NUMERO_DOCUMENTO
        WHERE a.TELEFONO IS NOT NULL

        UNION ALL

        SELECT a.NUMERO_DOCUMENTO, a.TELEFONO, a.TIPO_GESTION, a.GESTION, a.FECHA_ENVIO, 8 AS peso
        FROM [MAEBA].[ADM_OBJ_TG].tGestionMesEfectiva a
        INNER JOIN odin.dbo.ref_alfin_dni b
            ON a.NUMERO_DOCUMENTO = b.NUMERO_DOCUMENTO
        WHERE a.TELEFONO IS NOT NULL

        UNION ALL

        SELECT a.DNI_CLIENTE AS NUMERO_DOCUMENTO,a.Telefono_Llamado AS TELEFONO,a.ESTADOS AS TIPO_GESTION, a.Descripcion AS GESTION,a.FECHA_ENVIO, 7 AS peso
        FROM [MAEBA].[ADM_OBJ_TG].tGestionAgentesEfectivaN a
        INNER JOIN odin.dbo.ref_alfin_dni b
            ON a.DNI_CLIENTE = b.NUMERO_DOCUMENTO
        WHERE a.Telefono_Llamado IS NOT NULL

        UNION ALL

        SELECT a.CODDOC AS NUMERO_DOCUMENTO, a.TELEFONO, a.TIPO_GESTION, a.GESTION, a.FECHA_ENVIO, 5 AS peso
        FROM [MAEBA].[ADM_OBJ_TG].tGestionMesCencoPP a
        INNER JOIN odin.dbo.ref_alfin_dni b
            ON a.CODDOC = b.NUMERO_DOCUMENTO
        WHERE a.TELEFONO IS NOT NULL

        UNION ALL

        SELECT a.NUMERO_DOCUMENTO, a.TELEFONO, a.TIPO_GESTION, a.GESTION, a.fecha_llamada AS FECHA_ENVIO, 3 AS peso
        FROM [MAEBA].[ADM_OBJ_TG].tGestionMesDinerstc a
        INNER JOIN odin.dbo.ref_alfin_dni b
            ON a.NUMERO_DOCUMENTO = b.NUMERO_DOCUMENTO
        WHERE a.TELEFONO IS NOT NULL
    ) t
    """
df_maeba=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)


query = f"""
    SELECT *
        FROM (
            select 
            a.dni_cliente as NUMERO_DOCUMENTO,  
            a.celular as TELEFONO,
            a.Fecha as FECHA_ENVIO,
            b.nivel_1 AS TIPO_GESTION,
            b.nivel_2 AS GESTION ,2 as peso 
            FROM VALENTINA.dbo.alfin_gestion a
            inner join cronox.dbo.ref_alfin_dni C
            on a.dni_cliente COLLATE Modern_Spanish_CI_AS=C.NUMERO_DOCUMENTO   
            LEFT JOIN VALENTINA.dbo.alfin_tipificaciones B 
            ON A.Tipificacion = B.id_banco
            where a.fecha<'2026-06-01'
            UNION ALL
            select 
            a.dni_cliente as NUMERO_DOCUMENTO,  
            a.celular as TELEFONO,
            a.Fecha as FECHA_ENVIO,
            b.nivel_1 AS TIPO_GESTION,
            b.nivel_2 AS GESTION, 1 as peso 
            FROM VALENTINA.dbo.alfcc_gestion a
            inner join cronox.dbo.ref_alfin_dni C
            on a.dni_cliente COLLATE Modern_Spanish_CI_AS=C.NUMERO_DOCUMENTO   
            LEFT JOIN VALENTINA.dbo.alfcc_tipificaciones B 
            ON A.Tipificacion = B.id_banco
            where a.fecha<'2026-06-01'
        ) t
        """
df_valentina=obtener_tabla_sql(spark,query,server_sa,user_sa,pwd_sa,db_sa)
df_cet=df_maeba.unionByName(df_valentina)



window_spec = Window.partitionBy("NUMERO_DOCUMENTO").orderBy(F.col("peso").asc(),F.col("FECHA_ENVIO").desc())
df_cet = df_cet.withColumn("ref_01", row_number().over(window_spec))
df_cet = df_cet.filter(col("ref_01") == 1).drop('ref_01','peso')

In [ ]:
df_desembolso = df_desembolso.withColumn(
    "dif_meses",
    F.floor(
        F.months_between(
            F.to_date(F.lit('2026-05-01')),
            F.col("fecha_llamada")
        )
    )
)

df_desembolso=df_desembolso.withColumn('marca',when(F.col('dif_meses').isin(0,1),'INVENTARIO TARGET 0')
                                                .when(F.col('dif_meses').isin(2,3,4),'INVENTARIO TARGET 1')
                                                .otherwise(F.lit('INVENTARIO TARGET 2'))
)

df_gestion=df_gestion.filter(F.col('nivel_2').isin(nivel_2))
df_gestion=df_gestion.filter(F.col('nivel_1').isin('CONTACTO EFECTIVO CON TITULAR'))
df_gestion = df_gestion.withColumn(
    "dni_cliente",
    F.right(
        F.concat(F.lit("00000000"), F.col("dni_cliente")),
        F.lit(8)
    )

)

df_gestion=df_gestion.withColumn('marca',when(F.col('dif_meses').isin(0,1),'CET TARGET 0')
                                                .when(F.col('dif_meses').isin(2,3,4),'CET TARGET 1')
                                                .otherwise(F.lit('CET TARGET 2'))
)

In [ ]:

import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from sqlalchemy import create_engine
from sqlalchemy import text

import numpy as np

server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

engine_mysql = create_engine(
    f"mysql+pymysql://{user_envio}:{pwd_envio}@{server_envio}:{port_mysql}/{db_envio}"
)


In [3]:

filename='BASE AGOSTO TARGET.csv'
df_gestiones=cargar_archivo_csv(spark,filename,';',True)

filename='Gestiones TARGET (46).csv'
df_base=cargar_archivo_csv(spark,filename,';',True)

In [8]:
def completar_con_ceros(df, columna, longitud=8):
    return df.withColumn(
        columna,
        F.lpad(F.col(columna).cast("string"), longitud, "0")
    )




In [ ]:
df_gestiones=completar_con_ceros(df_gestiones,'Documento_RUC')

In [9]:
df_gestiones=df_gestiones.withColumnRenamed('Documento_RUC','DNI')
df_gestiones.show(2)

+--------+-------+--------------------+-----------+----------------+------+------+-----------+--------+-----------+-------+-------+------------------+----------+------------+------+------+------+---+---+---+----------+--------------+---------+----------+---------+
|     DNI|CARTERA|        NombreDeudor|      conca|         cosecha| Canal| Grupo|CodigoGrupo|Segmento|Subsegmento|     DK|     DT|       RANGO_DEUDA|antiguedad|fechaCastigo|    C1|    C2|    C3| C6|C12|C24|nvl_Asesor|nvl_Supervisor|nvl_Socio|CEF_INICIO|Percentil|
+--------+-------+--------------------+-----------+----------------+------+------+-----------+--------+-----------+-------+-------+------------------+----------+------------+------+------+------+---+---+---+----------+--------------+---------+----------+---------+
|42512978|     OH|PABLO CESAR CARRE...|42512978-OH|COSECHA OH 19/12|TARGET|TARGET|     TARGET|      S4|          B|1632.25|3215.73|04.  <1,000-2,000]|   8.<9-+>|   1/09/2017|737.61|409.61|273.08|  0|  0|  

In [ ]:
df_gestiones_base=df_gestiones.join(df_base,['DNI'],'left')

In [12]:
print(df_gestiones.join(df_base,['DNI'],'left').count())
print(df_gestiones.join(df_base,['DNI'],'leftanti').count())
print(df_base.join(df_gestiones,['DNI'],'leftanti').count())

58232
48378
4


In [10]:
print(df_gestiones.columns)
print(df_base.columns)
gestiones=set(df_gestiones.columns)
base=set(df_base.columns)

['DNI', 'CARTERA', 'NombreDeudor', 'conca', 'cosecha', 'Canal', 'Grupo', 'CodigoGrupo', 'Segmento', 'Subsegmento', 'DK', 'DT', 'RANGO_DEUDA', 'antiguedad', 'fechaCastigo', 'C1', 'C2', 'C3', 'C6', 'C12', 'C24', 'nvl_Asesor', 'nvl_Supervisor', 'nvl_Socio', 'CEF_INICIO', 'Percentil']
['Canal Gestión', 'Canal Asignación', 'DNI', 'Nombre Cliente', 'Cartera', 'Asesor', 'Equipo', 'Teléfono', 'Fecha Llamada', 'Campaña', 'Hora', 'Nivel 1', 'Nivel 2', 'Fecha Compromiso', 'Monto', 'Observación', 'Medio Gestión']


{'DNI'}

In [ ]:
filename='prueba_aaaa.csv'
df_lista=cargar_archivo_csv(spark,filename,';',True)
from pyspark.sql import functions as F


filename='Consulta_de_Campañas_202608_V3_SS (RED_CALL)db2.csv'
df_validar_01=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)
filename='Consulta_de_Campañas_202608_V3_SS (RED_CALL)db.csv'
df_validar_02=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)
filename='Consulta_de_Campañas_202608_V5_SS_EXT (CAMPO)_db2.csv'
df_validar_03=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)
filename='Consulta_de_Campañas_202608_V5_SS_EXT (CAMPO)_db.csv'
df_validar_04=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)
df_validar_01=df_validar_01.drop('TASA_MIN_DESCUENTO')
df_validar_02=df_validar_02.drop('TASA_MIN_DESCUENTO')
df_validar_01=df_validar_01.withColumn('tipo_archivo',F.lit('campo'))
df_validar_02=df_validar_02.withColumn('tipo_archivo',F.lit('campo'))
df_validar_03=df_validar_03.withColumn('tipo_archivo',F.lit('call'))
df_validar_04=df_validar_04.withColumn('tipo_archivo',F.lit('call'))

df_validar=df_validar_01.unionByName(df_validar_02).unionByName(df_validar_03).unionByName(df_validar_04)


In [35]:
fecha_mes_base='2026-08-01'
# tipi_cond1='diners'
servidor_01=21
tipi_cond1='plus'
tipi_cond2='PPD'
tipi_cond3='xx'
tb_tipolofia='tTipologia_Diners_PPD'
tipi_cod='cod'
tipi_resp_cod='C0'
tipi_descrip='[NIVEL 4]'
tipi_estado='[NIVEL 2]'
tipi_resp_estado='NO CONTACTO'
tipi_subdescripcion='[NIVEL 3]'
tnum_tb='tNumeroDiners'
tnum_dni='NumDoc'

tlista_generada='borrar_prestamo_diner'
get_base=since_base_maestra_pp_dinners

def resumen_vicidial(spark,fecha_mes_base,tipi_cond1,tipi_cond2,tipi_cond3,tb_tipolofia,servidor_01,tipi_cod,tipi_resp_cod,tipi_descrip,tipi_estado,tipi_resp_estado):
    query = f"""
        SELECT *
        FROM OPENQUERY([192.168.3.{servidor_01}], '
            SELECT        
            rtrim(ltrim(d.vendor_lead_code)) AS vendor_lead_code,        
            e.dial_method,
            a.campaign_id AS numero_campana,        
            a.user AS dni_ejecutivo,
            c.full_name AS ejecutivo,
            e.campaign_name AS nombre_campana,        
            a.call_date AS fecha_hora_llamada,        
            a.length_in_sec AS duracion,        
            b.status_name AS call_result,        
            f.list_description,        
            f.list_name,        
            a.phone_number as phone_number,        
            d.alt_phone as fecha_agenda,        
            d.comments as comentarios,        
            a.status AS codigo,
            a.term_reason,	
            a.alt_dial
            FROM asterisk.vicidial_log a         
            LEFT JOIN asterisk.vicidial_list d ON a.lead_id=d.lead_id        
            LEFT JOIN asterisk.vicidial_campaigns e ON a.campaign_id=e.campaign_id        
            LEFT JOIN asterisk.vicidial_lists f ON a.list_id=f.list_id        
            LEFT JOIN asterisk.vicidial_statuses b ON a.status=b.status        
            LEFT JOIN asterisk.vicidial_users c ON a.user=c.user        
            WHERE (e.campaign_name like "%{tipi_cond1}" or e.campaign_name like "%{tipi_cond2}" or e.campaign_name like "%{tipi_cond3}")
            AND a.call_date >= DATE_FORMAT(''{fecha_mes_base}'', ''%Y-%m-01'')
            AND a.call_date < 
            DATE_ADD(DATE_FORMAT(''{fecha_mes_base}'', ''%Y-%m-01''), INTERVAL 1 MONTH)
        ')

        """
    df_vicidial=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)

    df_vicidial = df_vicidial.withColumn(
        "vendor_lead_code",
        F.lpad(F.col("vendor_lead_code").cast("string"), 8, "0")
    )
    query = f"""
        SELECT {tipi_cod} as codigo
        , case
            when {tipi_cod}='CALLBK' then 'VOLVER A LLAMAR - call'
            else {tipi_descrip} 
        end as descripcion
        ,case 
            when {tipi_cod}='CALLBK' then 1200
            else peso 
        end as peso  FROM [ODIN].[dbo].{tb_tipolofia}
        where LEFT({tipi_cod},2)='{tipi_resp_cod}' or {tipi_estado}='{tipi_resp_estado}' or {tipi_cod}='CALLBK'
        """
    df_tipi=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)

    df_vicidial=df_vicidial.join(df_tipi,["codigo"],"left")

    return df_vicidial.select('fecha_hora_llamada','list_name','vendor_lead_code','phone_number','descripcion','dni_ejecutivo','ejecutivo','dial_method','term_reason','alt_dial','call_result','duracion','codigo','nombre_campana')

df_vici=resumen_vicidial(spark,fecha_mes_base,tipi_cond1,tipi_cond2,tipi_cond3,tb_tipolofia,servidor_01,tipi_cod,tipi_resp_cod,tipi_descrip,tipi_estado,tipi_resp_estado)
df_vici.filter(F.col('vendor_lead_code')=='06315016').orderBy(F.col('fecha_hora_llamada').desc()).show(truncate=False)


+------------------+---------+----------------+------------+-----------+-------------+---------+-----------+-----------+--------+-----------+--------+------+--------------+
|fecha_hora_llamada|list_name|vendor_lead_code|phone_number|descripcion|dni_ejecutivo|ejecutivo|dial_method|term_reason|alt_dial|call_result|duracion|codigo|nombre_campana|
+------------------+---------+----------------+------------+-----------+-------------+---------+-----------+-----------+--------+-----------+--------+------+--------------+
+------------------+---------+----------------+------------+-----------+-------------+---------+-----------+-----------+--------+-----------+--------+------+--------------+

